Common unility

In [64]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# from numba import njit
# from numba import jit

import matplotlib.pyplot as plt
from zigzag import *
import datetime

In [65]:
from tqsdk.ta import MA, EMA

In [66]:
from tqsdk.tafunc import ma, ema, abs, std, hhv, llv, count, time_to_datetime, barlast

In [67]:
from numpy import nan_to_num

# Functions

## get_long_orders

In [68]:
def get_long_order(source):
    data = source.copy()
# long only
    data['orders'] = 0
    position = 0
    for k in range(len(data)):
        if data.LONG[k]==1 and position == 0 :
            data.orders.values[k] = 1
            position = 1
        elif position == 1  and data.S_LONG[k]==1:
            data.orders.values[k] = -1
            position = 0
    return data

## get_short_orders

## backtest_long

In [69]:
def backtest_long(rb_df,init_cash):
    positions = pd.DataFrame(index = rb_df.index)
    portfolio = pd.DataFrame(index = rb_df.index)
    positions['stock'] = 4*rb_df['orders'].cumsum()
    # portfolio = positions.multiply(rb_df['close'],axis=0)
    portfolio['holding_values'] = positions.multiply(rb_df['close'],axis=0)
    pos_diff = positions.diff()
    portfolio['cash'] = init_cash - (pos_diff.multiply(rb_df['close'],axis=0)).cumsum()
    portfolio['total'] = portfolio['cash'] + portfolio['holding_values']
    portfolio['orders'] = rb_df['orders']
    portfolio['price']=rb_df.orders.diff()*rb_df.close
    return portfolio

In [70]:
# rb_df.orders.diff()*rb_df.close

## backtest_short

In [71]:
def backtest_short(rb_df,init_cash):
#     initial_cash = 20000
    positions = pd.DataFrame(index = rb_df.index)
    portfolio = pd.DataFrame(index = rb_df.index)
    positions['stock'] = 4*rb_df['orders'].cumsum()
    # portfolio = positions.multiply(rb_df['close'],axis=0)
    portfolio['holding_values'] = positions.multiply(rb_df['close'],axis=0)
    pos_diff = positions.diff()

    portfolio['cash'] = init_cash + (pos_diff.multiply(rb_df['close'],axis=0)).cumsum()
    portfolio['total'] = portfolio['cash'] - portfolio['holding_values']
    return portfolio


## get_trading_summary

In [72]:
def get_trading_summary(portfolio):
    df_only_orders = portfolio.query("orders in (1,-1)") 
    df_only_orders.loc[:,('lag_total')] = df_only_orders.total.shift(1)
    df_only_orders.loc[:,('lead_total')] = df_only_orders.total.shift(-1)
    df_only_orders.loc[:,('pro_loss')] = df_only_orders\
        .apply(lambda x:x.lead_total - x.total if x.orders==1 else  x.total -x.lag_total,axis=1)
    df_only_orders.loc[:,('date')] = df_only_orders.index.map(lambda x:x[:10])
    return df_only_orders.filter(['pro_loss','orders','price','date'])

## get_win_loss_ration

In [73]:
def get_win_loss_ration(trade_summary):
    df_win = trade_summary.query('pro_loss > 0 and orders == 1')
    df_loss = trade_summary.query('pro_loss < 0 and orders == 1')
    win_avg = df_win.pro_loss.mean()
    loss_avg = df_loss.pro_loss.mean()
    return {'win_avg':win_avg,'loss_avg':loss_avg,'ratio':win_avg/loss_avg}

## get_top1_vol

In [74]:
# def get_top1_vol(df):
#     return df.

## get_top2_vol

## get_top3_vol

## get_ma_stage pandas

In [75]:
def get_ma_stage(kline):
    if (kline.dtxt & (kline.mslp1s >= 1000) & (kline.bias1 >= 1)):
        return 'dtjg'  # 多头进攻
    elif kline.dtxt & (kline.mslp1s < 1000) & (kline.bias1 > 1):
        return 'dttz'  # 多头调整
    elif kline.dtxt & (kline.bias4 >= -0.5) & (kline.bias4 <= 1):
        return 'cai_ma4'  # 踩ma4
    elif kline.dtxt & (kline.bias3 >= -0.5) & (kline.bias3 <= 1):
        return 'cai_ma3'  # 踩ma3
    elif kline.dtxt & (kline.bias2 >= -0.5) & (kline.bias2 <= 1):
        return 'cai_ma2'  # 踩ma2
    elif kline.dtxt & (kline.bias1 >= -0.5) & (kline.bias1 <= 1):
        return 'cai_ma1'  # 踩ma1
    elif kline.dtxt & (kline.bias3 <= -0.5) & (kline.bias4 >= 1):
        return 'dzj34'  # 多中继34
    elif kline.dtxt & (kline.bias2 <= -0.5) & (kline.bias3 >= 1):
        return 'dzj23'  # 多中继23
    elif kline.dtxt & (kline.bias1 <= -0.5) & (kline.bias2 >= 1):
        return 'dzj12'  # 多中继12
    elif kline.dtxt & (kline.bias4 <= -0.5):
        return 'oversell'
    elif kline.ktxt & (kline.mslp1s <= 1000) & (kline.bias1 < -1):
        return 'ktjg'  # 空头进攻
    elif kline.ktxt & (kline.mslp1s > 1000) & (kline.bias1 < -1):
        return 'kttz'  # 空头调整
    elif kline.ktxt & (kline.bias4 <= 0.5) & (kline.bias4 >= -1):
        return 'chou_ma4'  # 抽ma4
    elif kline.ktxt & (kline.bias3 <= 0.5) & (kline.bias3 >= -1):
        return 'chou_ma3'  # 抽ma3
    elif kline.ktxt & (kline.bias2 <= 0.5) & (kline.bias2 >= -1):
        return 'chou_ma2'  # 抽ma2
    elif kline.ktxt & (kline.bias1 <= 0.5) & (kline.bias1 >= -1):
        return 'chou_ma1'  # 抽ma1
    elif kline.ktxt & (kline.bias3 >= 0.5) & (kline.bias4 <= -1):
        return 'kzj34'  # 空中继34
    elif kline.ktxt & (kline.bias2 >= 0.5) & (kline.bias3 <= -1):
        return 'kzj23'  # 空中继23
    elif kline.ktxt & (kline.bias1 >= 0.5) & (kline.bias2 <= -1):
        return 'kzj12'  # 空中继12
    elif kline.ktxt & (kline.bias4 >= 0.5):
        return 'overbuy'
    else:
        return 'others'


## get_price_pos

combined into get_price_stage

In [76]:
def get_price_pos(base):
#     baseC = base.copy()
#     for i in range(base.shape[1]):
#         baseC.iloc[:,i] = abs(base.iloc[:,i])
#     return np.argmin(baseC.values,axis=1)+1
    return np.abs(base.values).argmin(axis=1) + 1

## get_min_bias

combined into get_price_stage

In [77]:
def get_min_bias(base):
    x = base.values
#     x = base.copy()
#     index = base.iloc[:,-1:].copy()
    return x[np.arange(len(x)), np.abs(x).argmin(axis=1)]

In [78]:
# def get_price_stage(base):
#     if (abs(base.min_bias) < 0.5):
#         return 'cai_ma' 
# #     + str(base.price_pos)
# #     elif base.min_bias > 0.5:
# #         return 'dt_pos'
# #     elif base.min_bias <-0.5:
# #         return 'kt_pos'
#     else:
#         return 'others'
#     return 'cai' if (abs(base.min_bias) < 0.5) else 'others'
#     return abs(base.min_bias) < 0.5

## get_zig_col(res,label)

In [79]:
def get_zig_col(res,label):
    cols = res.columns.values
    new_cols = cols.copy()
    for i in list(range(len(cols))):
        new_cols[i]= cols[i]+str(label)
    return new_cols

In [80]:
# get_zig_col(res,1)

## get_nth_peak

In [4]:
def get_nth_peak(peak,n):
    peak_non_zero_mask = peak.notna()
    peak_non_zero_values = peak[peak_non_zero_mask]
    peak_shifted_values = pd.Series([np.nan]*n + peak_non_zero_values.tolist()[:-n], 
                              index=peak_non_zero_values.index)
    return peak.mask(peak.notna(), peak_shifted_values)

## get_nth_valley

In [3]:
def get_nth_valley(valley,n):
    valley_non_zero_mask = valley.notna()
    valley_non_zero_values = valley[valley_non_zero_mask]
    valley_shifted_values = pd.Series([np.nan]*n + valley_non_zero_values.tolist()[:-n], 
                              index=valley_non_zero_values.index)
    return valley.mask(valley.notna(), valley_shifted_values)

## get_ema

In [30]:
def get_ema(data,conf):
    klinesC = data.copy()
    klinesC['ma1'] = EMA(klinesC, conf['ma']['m1'])
    klinesC['ma2'] = EMA(klinesC, conf['ma']['m2'])
    klinesC['ma3'] = EMA(klinesC, conf['ma']['m3'])
    klinesC['ma4'] = EMA(klinesC, conf['ma']['m4'])
    klinesC['ma5'] = EMA(klinesC, conf['ma']['m5']) 
    klinesC['ma6'] = EMA(klinesC, conf['ma']['m6']) 
    klinesC['ma7'] = EMA(klinesC, conf['ma']['m7'])
    return klinesC[['ma1','ma2','ma3','ma4','ma5','ma6','ma7']]

# Numpy Functions

## rolling_window

In [81]:
def rolling_window(a,window):
    shape = a.shape[:-1] + (a.shape[-1] - window +1, window)
    strides = a.strides + (a.strides[-1],)
    return np.lib.stride_tricks.as_strided(a, shape=shape, strides=strides)

In [82]:
# data = np.arange(20)
# rolling_window(data,10).max(1)

## zig

In [83]:
# ZIG_STATE_START = 0
# ZIG_STATE_RISE = 1
# ZIG_STATE_FALL = 2

# def zig(x=0.055):
# #ts.set_token("此处放入tushare的token！！！")
# #pro = ts.pro_api()
# #df = pro.daily(ts_code="603297.SH")
# # print(list(df["close"]))
# # df = ts.get_hist_data(‘000069‘)
# # df = df[::-1]
# # 获取股票交易数据的Tushare的使用方法 - 蜗牛爬行ing - 博客园
# # https://www.cnblogs.com/DreamRJF/p/8660630.html
# # posted on 2018-03-28 15:18 蜗牛爬行ing
# #df = ts.get_k_data(‘000069‘)
# # df = ts.get_k_data(‘600535‘)
# # df = ts.get_k_data(‘512040‘) # 富国国信价值 etf 基金
# # df = ts.get_h_data(‘000051‘, index=True) # 上证180等权指数 index 参数必须指定为True

# df = ts.get_k_data(‘000051‘, index=True, start=‘2011-01-01‘) # 上证180等权指数 index 参数必须指定为True

# #df = ts.get_h_data(‘399106‘, index=True) # index 参数必须指定为True
# #df = ts.get_h_data(‘399106‘, index=True) #深圳综合指数
# #df = ts.get_k_data(‘399106‘, index=True) #深圳综合指数
# # df = ts.get_k_data(‘931052‘, index=True) # 中证国信价值指数， 不支持的指数
# # df = ts.get_k_data(‘hs300‘) # 支持主要的几个股票指数的历史行情
# # 股票代码，即6位数字代码，或者指数代码
# # (sh=上证指数 sz=深圳成指 hs300=沪深300指数
# # sz50=上证50 zxb=中小板 cyb=创业板)
# #df = df.reset_index(drop=True)
# # df = df.iloc[-100:]
# #x = 0.055

# k = df["close"]

# #d = df["trade_date"]

# d = df[‘date‘]

# # d = df.index

# #print(k)

# #print(d)

# # 循环前的变量初始化

# # 端点 候选点 扫描点 端点列表 拐点线列表 趋势状态

# peer_i = 0
# candidate_i = None
# scan_i = 0
# peers = [0]
# z = np.zeros(len(k))
# state = ZIG_STATE_START
# while True:
# #print(peers)

# scan_i += 1
# if scan_i == len(k) - 1:

# # 扫描到尾部
# if candidate_i is None:
# peer_i = scan_i
# peers.append(peer_i)

# else:

# if state == ZIG_STATE_RISE:
# if k[scan_i] >= k[candidate_i]:
    
# peer_i = scan_i
# peers.append(peer_i)

# else:

# peer_i = candidate_i
# peers.append(peer_i)
# peer_i = scan_i
# peers.append(peer_i)

# elif state == ZIG_STATE_FALL:
# if k[scan_i] <= k[candidate_i]:

# peer_i = scan_i
# peers.append(peer_i)

# else:

# peer_i = candidate_i
# peers.append(peer_i)
# peer_i = scan_i
# peers.append(peer_i)

# break

# if state == ZIG_STATE_START:
# if k[scan_i] >= k[peer_i] * (1 + x):

# candidate_i = scan_i
# state = ZIG_STATE_RISE

# elif k[scan_i] <= k[peer_i] * (1 - x):
# candidate_i = scan_i
# state = ZIG_STATE_FALL

# elif state == ZIG_STATE_RISE:
# if k[scan_i] >= k[candidate_i]:

# candidate_i = scan_i
# elif k[scan_i] <= k[candidate_i]*(1-x):

# peer_i = candidate_i
# peers.append(peer_i)
# state = ZIG_STATE_FALL
# candidate_i = scan_i

# elif state == ZIG_STATE_FALL:

# if k[scan_i] <= k[candidate_i]:
#     candidate_i = scan_i

# elif k[scan_i] >= k[candidate_i]*(1+x):
#     peer_i = candidate_i
#     peers.append(peer_i)
#     state = ZIG_STATE_RISE
#     candidate_i = scan_i

# #线性插值， 计算出zig的值

# for i in range(len(peers) - 1):
#     peer_start_i = peers[i]
#     peer_end_i = peers[i+1]
#     start_value = k[peer_start_i]
#     end_value = k[peer_end_i]

# a = (end_value - start_value)/(peer_end_i - peer_start_i)# 斜率
# for j in range(peer_end_i - peer_start_i +1):

# z[j + peer_start_i] = start_value + a*j
# print(u‘...转向点的阀值、个数、位置和日期...‘)

# print(x, len(peers))
# print(peers)

# dates = [d[i] for i in peers]
# print(dates)

# #print([k[i] for i in peers])
# #print(list(k))
# #print(list(z))

# plt.plot(z)

# zig(x=0.055)


## get_price_stage

In [ ]:
def get_price_stage(bias,df_ma):
    ma_cnt = bias.shape[1]
    col_indx=[0,1,2,3,4,5,6]
    arr_bias=bias.values
    arr_df_ma=df_ma.values
    ll=list(range(1,arr_bias.shape[1]))
    ll.extend([0])
    arr_bias2=arr_bias[:,ll]
    arr_comb= np.absolute(arr_bias)+np.absolute(arr_bias2) # 获得相邻df_ma的sum
    df_ma_abs = np.absolute(arr_df_ma)
    check = np.round((arr_comb[:,:-1] - df_ma_abs),decimals=2) #相邻df_ma的sum - df，为0的则表示价格在着2条ma之间
    b = np.array(np.where(check == 0)) # 找到价格所在的位置
    a = np.transpose(b)
    
    overlap_count=np.unique(a[:, 0], return_counts=True) #找到价格在ma中间的个数
    overlap_list=np.split(a[:, 1], np.unique(a[:, 0], return_index=True)[1][1:]) #列出所有  价格在ma的位置
    
    idx_overlap=overlap_count[0] #index_data
    index=np.arange(bias.shape[0])
    index.shape=(bias.shape[0],1) #创建和外部df一样的dim
    obj_arr=np.empty((bias.shape[0],1), dtype=object) #创建obj array来存arr 价格所在的ma位置  如果
    obj_arr_2d=np.hstack((index,obj_arr)) #
    ## 赋值 价格在ma的所有位置
    obj_arr_2d[idx_overlap.tolist(),1]=overlap_list 
    temp=np.zeros(bias.shape[0])
    temp.shape=(bias.shape[0],1)
    temp1=np.hstack((obj_arr_2d,temp))
    temp1[idx_overlap.tolist(),2]=overlap_count[1]  #数据对应的index
    idx_above_ma = (bias>0).all(axis=1)
    idx_below_ma = (bias<0).all(axis=1)
    temp1[idx_above_ma,2]=99
    temp1[idx_below_ma,2]=-99
    out_ma_ind=temp1[temp1[:,2] == 0] # 找到arr无值的
    price_pos=np.abs(bias.values).argmin(axis=1) #获取最小bias的位置
    min_bias=arr_bias[np.arange(bias.shape[0]), np.abs(arr_bias).argmin(axis=1)] #获取最小bias值
    return np.column_stack((temp1, price_pos, min_bias))[:,1:]
#     return arr_bias

In [84]:
# def get_price_stage(bias,df_ma):
#     ma_cnt = bias.shape[1]
#     if (s3==m1) & (m3==l1) & (ma_cnt==7):
#         col_indx=[0,1,2,4,5,7,8]
# #         arr_bias=bias.values[:,[0,1,2,4,5,7,8]]
#         arr_bias=bias.values
# #         arr_df_ma=df_ma.values[:,[0,1,3,4,6,7]]
#         arr_df_ma=df_ma.values
#     elif (s3==m1) and (m3!=l1) & (ma_cnt==8):
#         col_indx=[0,1,2,4,5,6,7,8]
#         arr_bias=bias.values[:,[0,1,2,4,5,6,7,8]]
#         arr_df_ma=df_ma.values[:,[0,1,3,4,5,6,7]]     
#     elif (s3!=m1) and (m3==l1) & (ma_cnt==8):
#         col_indx=[0,1,2,3,4,5,7,8]
#         arr_bias=bias.values[:,[0,1,2,3,4,5,7,8]]
#         arr_df_ma=df_ma.values[:,[0,1,2,3,4,6,7]]  
#     else:
#         col_indx=[0,1,2,3,4,5,6,7,8]
#         arr_bias=bias.values[:,[0,1,2,3,4,5,6,7,8]]
#         arr_df_ma=df_ma.values[:,[0,1,2,3,4,5,6,7]] 
# #    arr_2_df_ma=arr_df_ma[:2]
# #    arr_3_df_ma=arr_df_ma[:3]  
#     ll=list(range(1,arr_bias.shape[1]))
#     ll.extend([0])
#     arr_bias2=arr_bias[:,ll]
#     arr_comb= np.absolute(arr_bias)+np.absolute(arr_bias2) # 获得相邻df_ma的sum
#     df_ma_abs = np.absolute(arr_df_ma)
#     check = np.round((arr_comb[:,:-1] - df_ma_abs),decimals=2) #相邻df_ma的sum - df，为0的则表示价格在着2条ma之间
#     b = np.array(np.where(check == 0)) # 找到价格所在的位置
#     a = np.transpose(b)
    
#     overlap_count=np.unique(a[:, 0], return_counts=True) #找到价格在ma中间的个数
#     overlap_list=np.split(a[:, 1], np.unique(a[:, 0], return_index=True)[1][1:]) #列出所有  价格在ma的位置
    
#     idx_overlap=overlap_count[0] #index_data
#     index=np.arange(bias.shape[0])
#     index.shape=(bias.shape[0],1) #创建和外部df一样的dim
#     obj_arr=np.empty((bias.shape[0],1), dtype=object) #创建obj array来存arr 价格所在的ma位置  如果
#     obj_arr_2d=np.hstack((index,obj_arr)) #
#     ## 赋值 价格在ma的所有位置
#     obj_arr_2d[idx_overlap.tolist(),1]=overlap_list 
#     temp=np.zeros(bias.shape[0])
#     temp.shape=(bias.shape[0],1)
#     temp1=np.hstack((obj_arr_2d,temp))
#     temp1[idx_overlap.tolist(),2]=overlap_count[1]  #数据对应的index
#     idx_above_ma = (bias>0).all(axis=1)
#     idx_below_ma = (bias<0).all(axis=1)
#     temp1[idx_above_ma,2]=99
#     temp1[idx_below_ma,2]=-99
#     out_ma_ind=temp1[temp1[:,2] == 0] # 找到arr无值的
#     price_pos=np.abs(bias.values).argmin(axis=1) #获取最小bias的位置
#     min_bias=arr_bias[np.arange(bias.shape[0]), np.abs(arr_bias).argmin(axis=1)] #获取最小bias值
#     return np.column_stack((temp1, price_pos, min_bias))[:,1:]
# #     return arr_bias

## get_pct_freq

In [85]:
def get_pct_freq(pct_data,n,bin):
    col=15
    res=np.arange(col*8400)
# col 0 index，col 1开始存数据 1-10列用来存bin范围的值  
#     1-2 极端和异常下跌 3，4，5 大中小跌
#     6，7为震荡
#    8，9，10为小中大涨，11，12为异常和极端上涨 
#    13 存十字星  14列存震荡
    res.shape=(col,8400)
    res=res.T
    ll=len(bin)
    res[:,1:]=0
    for i in list(range(ll+1)):
        if i == 0:
#             col 1开始存数据
            res[:,i+1]=np.nan_to_num(count(pct_data<bin[i],n).values,nan=0)
        elif i==11:
            res[:,i+1]=np.nan_to_num(count(bin[i-1]<=pct_data,n).values,nan=0)
        else:
            res[:,i+1]=np.nan_to_num(count((bin[i-1] <= pct_data) & (pct_data < bin[i]),n).values,nan=0)
     # 将十字星单独拎出来
    res[:,ll+2]=np.nan_to_num(count(0==pct_data,n).values,nan=0)
    # 存震荡
    res[:,ll+3]=res[:,6] + res[:,7]
    return res

In [86]:
def get_pct_freq2(row):
    bin_idx=np.arange(12)
    nz=np.zeros(12)
    bin_base=np.vstack((bin_idx,nz)).T
    uniqueValues, occurCount = np.unique(row, return_counts=True)
#     bin_base[]
    res=np.asarray((uniqueValues, occurCount)).T
    bin_base[res[:,0].tolist(),1]=res[:,1]
    dw_part= bin_base[:6,1].sum()
    dw_pct = dw_part/row.shape[0]*100
#     net_up_pct
    return dw_pct #bin_base

In [87]:
# 比较耗时 300多ms
# [get_pct_freq(dd[i,:]) for i in list(range((dd.shape[0])))]

## get_pct_stage

In [88]:
def get_pct_stage(res):
# col 0 index，col 1开始存数据 1-10列用来存bin范围的值  
#     1-2 极端和异常下跌 3，4，5 大中小跌
#     6，7为震荡
#    8，9，10为小中大涨，11，12为异常和极端上涨 
#    13 存十字星  14列存震荡
    mtm=np.zeros((res.shape[0],20))
#     0-4 下跌  6-10 上涨
#    算下跌力
    mtm[:,0]=res[:,3]*2.5+res[:,4]*1.5+res[:,5]
    # 算上涨力
    mtm[:,1]=res[:,10]*2.5+res[:,9]*1.5+res[:,8]
    # 算净力
    mtm[:,2]=mtm[:,1] - mtm[:,0]
    #有波动 没有大跌 没有中跌 没有大涨 没有中涨 小涨 3
    pattern1=(mtm[:,1]>=3) & (mtm[:,2]>-2) & (mtm[:,2]<=2) & (res[:,3]==0) & (res[:,4]==0) & (res[:,10]==0) & (res[:,9]==0)
    mtm[pattern1,3]=1
    #有波动 没有大跌 没有中跌 没有大涨 中涨 1 小涨 1
    pattern2=(mtm[:,1]>=2.5) & (mtm[:,2]>-2) & (mtm[:,2]<=2) & (res[:,10]==0)  & (res[:,9]>0)
    mtm[pattern2,4]=2
    #有波动 有大涨 有中涨
    pattern3=(mtm[:,1]>=4) & (mtm[:,2]>-2) & (mtm[:,2]<=2) & (res[:,10]>0)  & (res[:,9]>0)
    mtm[pattern3,5]=3
    # 波动小 有小涨>=2 小跌=0
    pattern4=(mtm[:,1]>=2) & (mtm[:,0]==0) & (mtm[:,2]<=4) & (res[:,10]==0)  & (res[:,9]==0)
    mtm[pattern4,6]=4
    # 大阳后小波动
    
    
    return mtm
# 11 十字星    
# 10[inf,20)  极端波动
# 9 [20,5) 异常波动
# 8 [5,3) 大阳
# 7 [3,1.8]  中涨
# 6 [1.8,0.8]  小涨
# 5 [0.8,-0.8] 震荡
# 4 [-0.8, -1.8] 小跌
# 3 [-1.8, -3] 中跌
# 2 [-3.-3,5] 大跌
# 1 [-5,-20] 异常
# 0[-20, -inf]   
# pct_cnt
# 10[inf,20)  极端波动
# 9 [20,5) 异常波动
# 8 [5,3) 大阳
# 7 [3,1.8]  中涨
# 6 [1.8,0.8]  小涨
# 5 [0.8,-0.8] 震荡
# 4 [-0.8, -1.8] 小跌
# 3 [-1.8, -3] 中跌
# 2 [-3.-3,5] 大跌
# 1 [-5,-20] 异常
# 0 [-20, -inf]   

## get_3ma_stage

In [89]:
# 千分之1
unit=0.001
thresh1=1
thresh2=3

In [90]:
# def get_3ma_stage(ma_data,thresh1,thresh2):
#     unit = 0.001
#     res=np.zeros((ma_data.shape[0],20))
#     小涨=1 + thresh1*unit
#     大涨=1 + thresh2*unit
#     小跌=1 - thresh1*unit
#     大跌=1 - thresh2*unit
#     ma_ratio=pd.DataFrame(ma_data.values,columns=['ma12_ratio','ma23_ratio','ma13_ratio'])
    
# # 普通上涨 进攻 ma12_ratio > ma23_ratio >1 & ma13_ratio < 1.2
#     p1 = (1 <= ma_ratio.ma23_ratio) & (ma_ratio.ma23_ratio < ma_ratio.ma12_ratio) & \
#                (ma_ratio.ma13_ratio < 小涨)
#     res[p1,0] = 1
# # 中度上涨  ma12_ratio > ma23_ratio >1 & 1.2 <= ma13_ratio & ma13_ratio < 1.5
#     p2 = (1 <= ma_ratio.ma23_ratio) & (ma_ratio.ma23_ratio < ma_ratio.ma12_ratio) & \
#                 (小涨 <= ma_ratio.ma13_ratio) & (ma_ratio.ma13_ratio < 大涨)
#     res[p2,1] = 2
# # 高度上涨  ma12_ratio > ma23_ratio >1 & 1.5 <= ma13_ratio
#     p3 = (1 <= ma_ratio.ma23_ratio) & (ma_ratio.ma23_ratio < ma_ratio.ma12_ratio) & \
#                  (大涨 <= ma_ratio.ma13_ratio)
#     res[p3,2] = 3
# # 高度上涨放缓 ma23_ratio > ma12_ratio > 1 & (1.5 <= ma13_ratio) 
#     p4 = (1 <= ma_ratio.ma12_ratio) & (ma_ratio.ma12_ratio <= ma_ratio.ma23_ratio) & \
#                 (大涨 <= ma_ratio.ma13_ratio)
#     res[p4,3] = 4
    
# # 中度上涨放缓 ma23_ratio > ma12_ratio > 1 & (1.2 <= ma13_ratio) 
# #                   & (1.2 <= ma13_ratio) &(ma13_ratio < 1.5)
#     p5 = (1 <= ma_ratio.ma12_ratio) & (ma_ratio.ma12_ratio <= ma_ratio.ma23_ratio) & \
#                  (小涨 <= ma_ratio.ma13_ratio) & (ma_ratio.ma13_ratio < 大涨)  
#     res[p5,4] = 5
# # 普通上涨放缓 ma23_ratio > ma12_ratio > 1 & (ma13_ratio < 1.2)
#     p6 = (1 <= ma_ratio.ma12_ratio) & (ma_ratio.ma12_ratio <= ma_ratio.ma23_ratio) & \
#                  ( ma_ratio.ma13_ratio < 小涨)
#     res[p6,5] = 6
# # 中高上涨回调

#     p7 = (1 <= ma_ratio.ma23_ratio) & (ma_ratio.ma12_ratio < 1) & \
#                  (小涨 <= ma_ratio.ma13_ratio)
#     res[p7,6] = 7
# # 普通上涨回调
#     p8 = (1 <= ma_ratio.ma23_ratio) & (ma_ratio.ma12_ratio < 1) & \
#                  (1 <= ma_ratio.ma13_ratio) &  (ma_ratio.ma13_ratio <小涨)
#     res[p8,7] = 8
# # 多转空回调
#     p9 = (1 <= ma_ratio.ma23_ratio) & (ma_ratio.ma12_ratio < 1) & \
#                  (ma_ratio.ma13_ratio < 1)  
#     res[p9,8] = 9
    
# # 普通空头 ma12_ratio<ma23_ratio < 1
#     p10 = (ma_ratio.ma12_ratio <= ma_ratio.ma23_ratio) & (ma_ratio.ma23_ratio < 1) & \
#                  (小跌 <= ma_ratio.ma13_ratio)
#     res[p10,9] = 10
# # 中度空头 ma12_ratio < ma23_ratio < 1
#     p11 = (ma_ratio.ma12_ratio <= ma_ratio.ma23_ratio) & (ma_ratio.ma23_ratio < 1) & \
#                  (大跌 <= ma_ratio.ma13_ratio) & (ma_ratio.ma13_ratio < 小跌)
#     res[p11,10] = 11
# # 高度空头 ma12_ratio<ma23_ratio < 1        
#     p12 = (ma_ratio.ma12_ratio < ma_ratio.ma23_ratio) & (ma_ratio.ma23_ratio < 1) & \
#                 (ma_ratio.ma13_ratio < 大跌)
#     res[p12,11] = 12
# # 高度空头放缓
#     p13 = (ma_ratio.ma23_ratio < ma_ratio.ma12_ratio) & (ma_ratio.ma12_ratio < 1) & \
#                 (ma_ratio.ma13_ratio < 大跌)
#     res[p13,12] = 13
# # 中度空头放缓  
#     p14 = (ma_ratio.ma23_ratio < ma_ratio.ma12_ratio) & (ma_ratio.ma12_ratio < 1) & \
#                 (大跌 <= ma_ratio.ma13_ratio) & (ma_ratio.ma13_ratio < 小跌)
#     res[p14,13] = 14
# # 普通空头放缓    
#     p15 = (ma_ratio.ma23_ratio < ma_ratio.ma12_ratio) & (ma_ratio.ma12_ratio < 1) & \
#                 (小跌 <= ma_ratio.ma13_ratio)
#     res[p15,14] = 15

# # 空头反弹
#     p16 = (1 <= ma_ratio.ma12_ratio) & (ma_ratio.ma23_ratio < 1) & \
#                 (ma_ratio.ma13_ratio < 小跌)
#     res[p16,15] = 16
    
# # 反转进攻  
#     p17 = (1 <= ma_ratio.ma12_ratio) & (ma_ratio.ma23_ratio < 1) & \
#                 (小跌 <= ma_ratio.ma13_ratio) & (ma_ratio.ma13_ratio < 1)
#     res[p17,16] = 17
    
# # 空转多反弹
#     p18 = (1 <= ma_ratio.ma12_ratio) & (ma_ratio.ma23_ratio < 1) & \
#                  (1 <= ma_ratio.ma13_ratio)
#     res[p18,17] = 18
#     res[:,18] = res[:,:18].sum(axis = 1)
#     res[1:,19] = res[0:-1,18]
#     return res

In [91]:
# chk_res = [[i,ma_res[:,i].sum()/(i+1)] for i in list(range(18))]

In [92]:
# chk_res

In [93]:
def get_ma_stage2(bias):
    window=3
    step=1
    
    index_list=np.arange(bias.shape[0])
    
    if (s3==m1) & (m3==l1) :
        bias_1_7=bias.values[:,[0,1,2,4,5,7,8]]
        col_len=bias_1_7.shape[1]
        iteration=int((col_len - window+1)/step)
        col_list=list(range(col_len))
        
        res=np.zeros((bias.shape[0],iteration))
        rng=np.zeros((bias.shape[0],iteration))
        
        for i in list(range(iteration)):
            start=i
            end=i+window-1
            cursor=col_list[start:end]
            bias_slice=bias_1_7[:,cursor]
            #获取最小bias
            min_bias_slice=bias_slice[index_list,bias_slice.argmin(axis=1)]
            #获取最大bias
            max_bias_slice=bias_slice[index_list,bias_slice.argmax(axis=1)]
            long=(min_bias_slice>=0)
            short=(min_bias_slice<0)
#             存多线的宽度
            rng[:,i]=max_bias_slice - min_bias_slice
            #多头
            res[long,i]=1
#             空头
            res[short,i]=-1

    elif (s3==m1) and (m3!=l1):
        col_indx=[0,1,2,4,5,6,7,8]
        arr_bias=bias.values[:,[0,1,2,4,5,6,7,8]]
#         arr_df_ma=df_ma.values[:,[0,1,3,4,5,6,7]]     

    elif (s3!=m1) and (m3==l1):
        col_indx=[0,1,2,3,4,5,7,8]
        arr_bias=bias.values[:,[0,1,2,3,4,5,7,8]]
#         arr_df_ma=df_ma.values[:,[0,1,2,3,4,6,7]]  
    else:
        col_indx=[0,1,2,3,4,5,6,7,8]
        arr_bias=bias.values[:,[0,1,2,3,4,5,6,7,8]]
#         arr_df_ma=df_ma.values[:,[0,1,2,3,4,5,6,7]] 
    return res,rng
  
    
# 解析数据 rng 
# rng[:,0] ma1,2,3的range
# rng[:,1] ma2,3,4的range
# 以此类推

#         bias_2_4=bias.values[:,[1,2,4]]
#         bias_3_5=bias.values[:,[2,4,5]]
#         bias_4_6=bias.values[:,[4,5,7]]
#         bias_5_7=bias.values[:,[5,7,8]]
        
#         min_bias_1_3=bias_1_3[index_list,bias_1_3.argmin(axis=1)]
#         min_bias_2_4=bias_2_4[index_list,bias_2_4.argmin(axis=1)]
#         min_bias_3_5=bias_3_5[index_list,bias_3_5.argmin(axis=1)]
#         min_bias_4_6=bias_5_7[index_list,bias_4_6.argmin(axis=1)]
#         min_bias_5_7=bias_5_7[index_list,bias_5_7.argmin(axis=1)]
#         min_bias_1_7=bias_1_6[index_list,bias_1_7.argmin(axis=1)]
        
        # ma1,2,3 多头
#         long_1_3=(min_bias_1_3>=0)
#         # ma1,2,3空头
#         short_1_3=(min_bias_1_3<0)
#         res[long_1_3,0]=1
#         res[short_1_3,0]=-1
        
# #         long_1_4=(min_bias_1_4>=0)
# #         short_1_4=(min_bias_1_4<0)
# #         res[long_1_4,1]=1
# #         res[short_1_4,1]=-1
        
#         long_2_4=(min_bias_2_4>=0)
#         short_2_4=(min_bias_2_4<0)
#         res[long_2_4,1]=1
#         res[short_2_4,1]=-1
        
#         long_3_5=(min_bias_3_5>=0)
#         short_3_5=(min_bias_3_5<0)
#         res[long_3_5,1]=1
#         res[short_3_5,1]=-1       

#         long_4_6=(min_bias_4_6>=0)
#         short_4_6=(min_bias_4_6<0)
#         res[long_4_6,1]=1
#         res[short_4_6,1]=-1   
        
#         long_5_7=(min_bias_5_7>=0)
#         short_5_7=(min_bias_5_7<0)
#         res[long_5_7,1]=1
#         res[short_5_7,1]=-1   

#         long_1_7=(min_bias_1_7>=0)
#         short_1_7=(min_bias_1_7<0)
#         res[long_1_7,1]=1
#         res[short_1_7,1]=-1   

## get_3ma_category

In [94]:
def get_3ma_category(ma_data):
    ma_ratio=ma_data.copy()
    ma_ratio=pd.DataFrame(ma_data.values,columns=['ma12_ratio','ma23_ratio','ma13_ratio'])

#     res=np.zeros((ma_ratio.shape[0],6))
    # 多转空 6
    p6 = (ma_ratio.ma12_ratio < 1)  & (ma_ratio.ma23_ratio >=1) & (ma_ratio.ma13_ratio <1)
    ma_ratio.loc[p6,'ma_category'] = 6
    # # 多头震荡 5
    p5 = (ma_ratio.ma12_ratio < 1)  & (ma_ratio.ma23_ratio >=1) & (ma_ratio.ma13_ratio >=1)
    ma_ratio.loc[p5,'ma_category'] = 5
    # # 多头 4
    p4 = (ma_ratio.ma12_ratio >= 1) & (ma_ratio.ma23_ratio >=1)
    ma_ratio.loc[p4,'ma_category'] = 4
    # 空转多 3
    p3 = (ma_ratio.ma12_ratio>=1) & (ma_ratio.ma23_ratio < 1) & (ma_ratio.ma13_ratio >= 1)
    ma_ratio.loc[p3,'ma_category'] = 3
    # 空头震荡 2
    p2 = (ma_ratio.ma12_ratio>=1) & (ma_ratio.ma23_ratio < 1) & (ma_ratio.ma13_ratio <1)
    ma_ratio.loc[p2,'ma_category'] = 2
    # 空头 1
    p1 = (ma_ratio.ma12_ratio < 1)  & (ma_ratio.ma23_ratio < 1) & (ma_ratio.ma13_ratio <1)
    ma_ratio.loc[p1,'ma_category'] = 1
    return ma_ratio[['ma_category']]

## get_3ma_rng_category

In [95]:
def get_3ma_rng_category(ma_data,kl_s_ratio,kl_m_ratio):
    ma_data=ma_data.copy()

    # 靠拢
    kl13= ma_data.rng_ma13 <= ma_data.ma_body * kl_s_ratio
    kl35 = ma_data.rng_ma35 <= ma_data.ma_body * kl_m_ratio
    
    ma_data.loc[kl13,'kl_ma13'] = 1
    ma_data.loc[kl35,'kl_ma35'] = 1
    
    #  均线发散
    ma_data.loc[~kl13,'kl_ma13'] = 0
    ma_data.loc[~kl35,'kl_ma35'] = 0
    
    return ma_data[['kl_ma13','kl_ma35']]

## get_vol_freq

In [96]:
def cal_vol_freq(vdata,n):
    res=np.zeros((vdata.shape[0],6))
    #统计极端量 》5
    res[:,0]=nan_to_num(count(5<vdata.vol3_strength,n),nan=0)
    #统计10天 巨量 2.8 < vol3_strength < 5
    res[:,1]=nan_to_num(count((2.8<vdata.v3_strength) & (vdata.v3_strength<=5),n),nan=0)
    #统计10天 大量 v3_strength [1.9,2.8]
    res[:,2]=nan_to_num(count((1.90<vdata.v3_strength) & (vdata.v3_strength<=2.8),n),nan=0)
    #统计10天 放量  【1.2， 1.9】
    res[:,3]=nan_to_num(count((1.20<vdata.v3_strength) & (vdata.v3_strength<=1.9),n),nan=0)
    #     正常量 【0.85，1.2】
    res[:,4]=nan_to_num(count((0.85<vdata.v3_strength) & (vdata.v3_strength<=1.2),n),nan=0)
    #  缩量 【0.5，0.85】
    res[:,5]=nan_to_num(count((0.5<vdata.v3_strength) & (vdata.v3_strength<=0.85),n),nan=0)
    #  严重缩量  【0，0.5】
    res[:,5]=nan_to_num(count((vdata.v3_strength<=0.5),n),nan=0)
    return res

In [97]:
# def get_mv_stage(vdata):
#     res=np.zeros((vdata.shape[0],18))
# # 普通放量 进攻 mv12_ratio > mv23_ratio >1 & mv13_ratio < 1.2
#     res[:,0] = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
#                (vdata.mv13_ratio < 1.2)
# # 中度放量   mv12_ratio > mv23_ratio >1 & 1.2 <= mv13_ratio & mv13_ratio < 1.5
#     res[:,1] = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
#                 (1.2 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 1.5)
# # 高度放量  mv12_ratio > mv23_ratio >1 & 1.5 <= mv13_ratio
#     res[:,2] = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
#                  (1.5 <= vdata.mv13_ratio)
# # 普通放量放缓 mv23_ratio > mv12_ratio > 1 & (mv13_ratio < 1.2)
#     res[:,3] = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
#                  ( vdata.mv13_ratio <1.2 )
# # 中度放量放缓 mv23_ratio > mv12_ratio > 1 & (1.2 <= mv13_ratio) 
# #                   & (1.2 <= mv13_ratio) &(mv13_ratio < 1.5)
#     res[:,4] = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
#                  (1.2 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 1.5)  
# # 高度放量放缓 mv23_ratio > mv12_ratio > 1 & (mv13_ratio < 1.5) 
#     res[:,5] = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
#                 (1.5 <= vdata.mv13_ratio)
# # 普通缩量 mv12_ratio<mv23_ratio < 1
#     res[:,6] = (vdata.mv12_ratio <= vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
#                  (0.85 <= vdata.mv13_ratio)
# # 中度缩量 mv12_ratio<mv23_ratio < 1
#     res[:,7] = (vdata.mv12_ratio <= vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
#                  (0.65 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 0.85)
# # 高度缩量 mv12_ratio<mv23_ratio < 1        
#     res[:,8] = (vdata.mv12_ratio < vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
#                 (vdata.mv13_ratio < 0.65)
# # 普通缩量放缓    
#     res[:,9] = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
#                 (0.85 <= vdata.mv13_ratio)
# # 中度缩量放缓  
#     res[:,10] = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
#                 (0.65 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 0.85)
# # 高度缩量放缓
#     res[:,11] = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
#                 (vdata.mv13_ratio < 0.65)
# # 缩量进攻 
#     res[:,12] = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
#                 (vdata.mv13_ratio < 0.85)
# # 反转进攻  
#     res[:,13] = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
#                 (0.85 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 1)
# # 放量进攻
#     res[:,14] = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
#                  (1 <= vdata.mv13_ratio)
# # 放量人气降低
#     res[:,15] = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
#                  (1.2 <= vdata.mv13_ratio)
# # 普通放量人气降低
#     res[:,16] = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
#                  (1 <= vdata.mv13_ratio) &  (vdata.mv13_ratio <1.2)
# # 骤降人气
#     res[:,17] = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
#                  (vdata.mv13_ratio < 1)  
#     return res

## get_yc_vol_price

In [120]:
def get_yc_vol_price(base,factor,thresh):
    kline=base[['close']].copy()
#     if op == 'gt':
    cond = factor > thresh
#     else: op == 'lt':
#         cond = factor < thresh
    yc_vol_p_temp = kline.close * cond
    yc_vol_p_temp.replace(0,np.nan, inplace=True)
#     yc_vol_p = yc_vol_p_temp.ffill()
    return yc_vol_p_temp.ffill()

    

## classify_v1_strength

In [99]:
def classify_v1_strength(base,thresh):
    data = base[['v1_strength']].copy()
    #     缩量
    data['vol_strength1_category'] = 0
    sl_idx = data.v1_strength < (1-thresh)
    data.loc[sl_idx,'vol_strength1_category'] = 1
    fl_idx = data.v1_strength >= (1+thresh)
    data.loc[fl_idx,'vol_strength1_category']=3
    data['vol_strength1_category'].replace(0,2,inplace=True)
    return data['vol_strength1_category']

In [100]:
thresh=0.25

## get_mv_stage

In [9]:
def get_mv_stage(data, vThresh):
    """
    基于3条均量，判断是否缩量，放量，还是平量
    cl: 常量 2
    fl: 放量 3
    sl: 缩量 1
    """
    vdata = data.copy()
    
#     vdata.loc[:,['cl_vol']] = ((1 - vThresh) <= vdata.mv12_ratio) & (vdata.mv12_ratio <= (1 + vThresh)) &\
#                         ((1 - vThresh) <= vdata.mv13_ratio) & (vdata.mv13_ratio <= (1 + vThresh)) & \
#                         ((1 - vThresh) <= vdata.mv23_ratio) & (vdata.mv23_ratio <= (1 + vThresh))
    vdata.loc[:,['sl_vol']] = (vdata.mv12_ratio < (1 - vThresh)) | (vdata.mv13_ratio < (1 - vThresh)) | (vdata.mv23_ratio < (1 - vThresh))
    vdata.loc[:,['fl_vol']] = (vdata['sl_vol'] == 0) & (((1 + vThresh) < vdata.mv12_ratio) | ((1 + vThresh) < vdata.mv13_ratio) | \
                                           ((1 + vThresh) < vdata.mv23_ratio ))
    vdata.loc[:,['cl_vol']] = (vdata['sl_vol'] == 0) & (vdata['fl_vol'] == 0)
    vdata.loc[:,['mv_stage']] = vdata.cl_vol*2 + vdata.sl_vol + vdata.fl_vol*3
    return vdata[['mv_stage']]  #'cl_vol','sl_vol','fl_vol',

In [102]:
# def get_mv_stage(vdata,thresh1,thresh2):
#     小涨 = thresh1
#     大涨 = thresh2
#     小跌 = round(1/thresh1,2)
#     大跌 = round(1/thresh2,2)
#     res=np.zeros((vdata.shape[0],20))
# # 普通放量 进攻 mv12_ratio > mv23_ratio >1 & mv13_ratio < 1.2
#     p1 = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
#                (vdata.mv13_ratio < 1.2)
#     res[p1,0] = 1
# # 中度放量   mv12_ratio > mv23_ratio >1 & 1.2 <= mv13_ratio & mv13_ratio < 1.5
#     p2 = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
#                 (1.2 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 1.5)
#     res[p2,1] = 2
# # 高度放量  mv12_ratio > mv23_ratio >1 & 1.5 <= mv13_ratio
#     p3 = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
#                  (1.5 <= vdata.mv13_ratio)
#     res[p3,2] = 3

# # 高度放量放缓 mv23_ratio > mv12_ratio > 1 & (mv13_ratio < 1.5) 
#     p4 = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
#                 (1.5 <= vdata.mv13_ratio)
#     res[p4,3] = 4
# # 中度放量放缓 mv23_ratio > mv12_ratio > 1 & (1.2 <= mv13_ratio) 
# #                   & (1.2 <= mv13_ratio) &(mv13_ratio < 1.5)
#     p5 = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
#                  (1.2 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 1.5)  
#     res[p5,4] = 5 
# # 普通放量放缓 mv23_ratio > mv12_ratio > 1 & (mv13_ratio < 1.2)
#     p6 = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
#                  ( vdata.mv13_ratio <1.2 )
#     res[p6,5] = 6
# # 中高放量回调
#     p7 = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
#                  (1.2 <= vdata.mv13_ratio)
#     res[p7,6] = 7
# # 普通放量回调
#     p8 = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
#                  (1 <= vdata.mv13_ratio) &  (vdata.mv13_ratio <1.2)
#     res[p8,7] = 8
# #  放转缩回调
#     p9 = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
#                  (vdata.mv13_ratio < 1)  
#     res[p9,8] = 9
# # 普通缩量 mv12_ratio<mv23_ratio < 1
#     p10 = (vdata.mv12_ratio <= vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
#                  (0.85 <= vdata.mv13_ratio)
#     res[p10,9] = 10
# # 中度缩量 mv12_ratio<mv23_ratio < 1
#     p11 = (vdata.mv12_ratio <= vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
#                  (0.65 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 0.85)
#     res[p11,10] = 11
# # 高度缩量 mv12_ratio<mv23_ratio < 1        
#     p12 = (vdata.mv12_ratio < vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
#                 (vdata.mv13_ratio < 0.65)
#     res[p12,11] = 12
# # 高度缩量放缓
#     p13 = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
#                 (vdata.mv13_ratio < 0.65)
#     res[p13,12] = 13
# # 中度缩量放缓  
#     p14 = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
#                 (0.65 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 0.85)
#     res[p14,13] = 14
# # 普通缩量放缓    
#     p15 = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
#                 (0.85 <= vdata.mv13_ratio)
#     res[p15,14] = 15
# # 中高缩量反弹
#     p16 = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
#                 (vdata.mv13_ratio < 0.85)
#     res[p16,15] = 16
# # 普通缩量反弹
#     p17 = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
#                 (0.85 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 1)
#     res[p17,16] = 17
# # 缩量转放量反转
#     p18 = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
#                  (1 <= vdata.mv13_ratio)
#     res[p18,17] = 18
#     res[:,18] = res[:,:19].sum(axis=1)
#     res[1:,19] = res[0:-1,18]
#     return res

In [103]:
# def get_mv_stage(vdata,thresh1,thresh2):
#     小涨 = thresh1
#     大涨 = thresh2
#     小跌 = round(1/thresh1,2)
#     大跌 = round(1/thresh2,2)
#     res=np.zeros((vdata.shape[0],20))
    
#     p1 = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
#                (vdata.mv13_ratio < 小涨)
#     res[p1,0] = 1
    
# # 中度放量   mv12_ratio > mv23_ratio >1 & 1.2 <= mv13_ratio & mv13_ratio < 1.5
#     p2 = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
#                 (小涨 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 大涨)
#     res[p2,1] = 2
# # 高度放量  mv12_ratio > mv23_ratio >1 & 大涨 <= mv13_ratio
#     p3 = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
#                  (大涨 <= vdata.mv13_ratio)
#     res[p3,2] = 3

# # 高度放量放缓 mv23_ratio > mv12_ratio > 1 & (mv13_ratio < 大涨) 
#     p4 = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
#                 (大涨 <= vdata.mv13_ratio)
#     res[p4,3] = 4
# # 中度放量放缓 mv23_ratio > mv12_ratio > 1 & (1.2 <= mv13_ratio) 
# #                   & (小涨 <= mv13_ratio) &(mv13_ratio < 大涨)
#     p5 = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
#                  (小涨 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 大涨)  
#     res[p5,4] = 5 
# # 普通放量放缓 mv23_ratio > mv12_ratio > 1 & (mv13_ratio < 小涨)
#     p6 = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
#                  ( vdata.mv13_ratio <小涨 )
#     res[p6,5] = 6
# # 中高放量回调
#     p7 = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
#                  (小涨 <= vdata.mv13_ratio)
#     res[p7,6] = 7
# # 普通放量回调
#     p8 = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
#                  (1 <= vdata.mv13_ratio) &  (vdata.mv13_ratio <小涨)
#     res[p8,7] = 8
# #  放转缩回调
#     p9 = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
#                  (vdata.mv13_ratio < 1)  
#     res[p9,8] = 9
# # 普通缩量 mv12_ratio<mv23_ratio < 1
#     p10 = (vdata.mv12_ratio <= vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
#                  (小跌 <= vdata.mv13_ratio)
#     res[p10,9] = 10
# # 中度缩量 mv12_ratio<mv23_ratio < 1
#     p11 = (vdata.mv12_ratio <= vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
#                  (大跌 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 小跌)
#     res[p11,10] = 11
# # 高度缩量 mv12_ratio<mv23_ratio < 1        
#     p12 = (vdata.mv12_ratio < vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
#                 (vdata.mv13_ratio < 大跌)
#     res[p12,11] = 12
# # 高度缩量放缓
#     p13 = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
#                 (vdata.mv13_ratio < 大跌)
#     res[p13,12] = 13
# # 中度缩量放缓  
#     p14 = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
#                 (大跌 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 小跌)
#     res[p14,13] = 14
# # 普通缩量放缓    
#     p15 = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
#                 (小跌 <= vdata.mv13_ratio)
#     res[p15,14] = 15
# # 中高缩量反弹
#     p16 = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
#                 (vdata.mv13_ratio < 小跌)
#     res[p16,15] = 16
# # 普通缩量反弹
#     p17 = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
#                 (小跌 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 1)
#     res[p17,16] = 17
    
# # 缩量转放量反转
#     p18 = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
#                  (1 <= vdata.mv13_ratio)
#     res[p18,17] = 18
    
#     res[:,18] = res[:,:19].sum(axis=1)
#     res[1:,19] = res[0:-1,18]
#     return res

In [104]:
def get_mv_stage2(vdata):
    res=np.zeros((vdata.shape[0],20))
# 普通放量 进攻 mv12_ratio > mv23_ratio >1 & mv13_ratio < 1.2
    p1 = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio)
#                (vdata.mv13_ratio < 1.2)
    res[p1,0] = 1

# 高度放量放缓 mv23_ra tio > mv12_ratio > 1 & (mv13_ratio < 1.5) 
    p2 = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio)
#     & \                (1.5 <= vdata.mv13_ratio)
    res[p2,1] = 2

# 中高放量回调
    p3 = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) 
#     & \         (1.2 <= vdata.mv13_ratio)
    res[p3,2] = 3

# 普通缩量 mv12_ratio<mv23_ratio < 1
    p4 = (vdata.mv12_ratio <= vdata.mv23_ratio) & (vdata.mv23_ratio < 1) 
#     & \        (0.85 <= vdata.mv13_ratio)
    res[p4,3] = 4
    
# 高度缩量放缓
    p5 = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) 
#     & \      (vdata.mv13_ratio < 0.65)
    res[p5,4] = 5
    
# 中高缩量反弹
    p6 = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) 
#     & \   (vdata.mv13_ratio < 0.85)
    res[p6,5] = 6
    res[:,6] = res[:,:6].sum(axis=1)
    res[1:,19] = res[0:-1,18]
    return res

In [105]:
def get_mv_stage_detail(vdata,thresh1,thresh2):
    小涨 = thresh1
    大涨 = thresh2
    小跌 = round(1/thresh1,2)
    大跌 = round(1/thresh2,2)
    res=np.zeros((vdata.shape[0],20))
    
    p1 = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
               (vdata.mv13_ratio < 小涨)
    res[p1,0] = 1
    
# 中度放量   mv12_ratio > mv23_ratio >1 & 1.2 <= mv13_ratio & mv13_ratio < 1.5
    p2 = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
                (小涨 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 大涨)
    res[p2,1] = 2
# 高度放量  mv12_ratio > mv23_ratio >1 & 大涨 <= mv13_ratio
    p3 = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) & \
                 (大涨 <= vdata.mv13_ratio)
    res[p3,2] = 3

# 高度放量放缓 mv23_ratio > mv12_ratio > 1 & (mv13_ratio < 大涨) 
    p4 = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
                (大涨 <= vdata.mv13_ratio)
    res[p4,3] = 4
# 中度放量放缓 mv23_ratio > mv12_ratio > 1 & (1.2 <= mv13_ratio) 
#                   & (小涨 <= mv13_ratio) &(mv13_ratio < 大涨)
    p5 = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
                 (小涨 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 大涨)  
    res[p5,4] = 5 
# 普通放量放缓 mv23_ratio > mv12_ratio > 1 & (mv13_ratio < 小涨)
    p6 = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio) & \
                 ( vdata.mv13_ratio <小涨 )
    res[p6,5] = 6
# 中高放量回调
    p7 = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
                 (小涨 <= vdata.mv13_ratio)
    res[p7,6] = 7
# 普通放量回调
    p8 = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
                 (1 <= vdata.mv13_ratio) &  (vdata.mv13_ratio <小涨)
    res[p8,7] = 8
#  放转缩回调
    p9 = (1 <= vdata.mv23_ratio) & (vdata.mv12_ratio < 1) & \
                 (vdata.mv13_ratio < 1)  
    res[p9,8] = 9
# 普通缩量 mv12_ratio<mv23_ratio < 1
    p10 = (vdata.mv12_ratio <= vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
                 (小跌 <= vdata.mv13_ratio)
    res[p10,9] = 10
# 中度缩量 mv12_ratio<mv23_ratio < 1
    p11 = (vdata.mv12_ratio <= vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
                 (大跌 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 小跌)
    res[p11,10] = 11
# 高度缩量 mv12_ratio<mv23_ratio < 1        
    p12 = (vdata.mv12_ratio < vdata.mv23_ratio) & (vdata.mv23_ratio < 1) & \
                (vdata.mv13_ratio < 大跌)
    res[p12,11] = 12
# 高度缩量放缓
    p13 = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
                (vdata.mv13_ratio < 大跌)
    res[p13,12] = 13
# 中度缩量放缓  
    p14 = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
                (大跌 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 小跌)
    res[p14,13] = 14
# 普通缩量放缓    
    p15 = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1) & \
                (小跌 <= vdata.mv13_ratio)
    res[p15,14] = 15
# 中高缩量反弹
    p16 = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
                (vdata.mv13_ratio < 小跌)
    res[p16,15] = 16
# 普通缩量反弹
    p17 = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
                (小跌 <= vdata.mv13_ratio) & (vdata.mv13_ratio < 1)
    res[p17,16] = 17
    
# 缩量转放量反转
    p18 = (1 <= vdata.mv12_ratio) & (vdata.mv23_ratio < 1) & \
                 (1 <= vdata.mv13_ratio)
    res[p18,17] = 18
    
    res[:,18] = res[:,:19].sum(axis=1)
    res[1:,19] = res[0:-1,18]
    return res

In [106]:
# def get_mv_stage2(vdata):
#     res=np.zeros((vdata.shape[0],20))
# #  普通放量 进攻 mv12_ratio > mv23_ratio >1 & mv13_ratio < 1.2
#     res[:,0] = (1 <= vdata.mv23_ratio) & (vdata.mv23_ratio <= vdata.mv12_ratio) 
# # 普通放量放缓 mv23_ratio > mv12_ratio > 1 & (mv13_ratio < 1.2)
#     res[:,1] = (1 <= vdata.mv12_ratio) & (vdata.mv12_ratio <= vdata.mv23_ratio)
#     res[:,2] = ( vdata.mv12_ratio <1 ) & (1 <= vdata.mv23_ratio) 
#     res[:,3] = ( vdata.mv23_ratio <1 ) & (1 <= vdata.mv12_ratio) 
#     res[:,4] = (vdata.mv23_ratio <= vdata.mv12_ratio) & (vdata.mv12_ratio < 1)
#     res[:,5] = (vdata.mv12_ratio <= vdata.mv23_ratio) & (vdata.mv23_ratio < 1)
#     res[:,6] = res[:,:6].sum(axis=1)
#     return res

In [107]:
# list(zip(chk_res,vol_mapping))

In [108]:
#     vol_data = klinesC
# [['volume','vol_prev','mv1','mv2','mv3','vol1_strength','vol2_strength','vol3_strength','mv13_ratio','mv23_ratio','mv12_ratio']]

def get_vol_stage(vdata):
    res=np.zeros((vdata.shape[0],20))
    res[:,:6]=cal_vol_freq(vdata,10)
    res[:,6:12]=cal_vol_freq(vdata,20)
    #统计40天 巨量 vol3_strength> 3
    #统计40天 大量 vol3_strength [1.6,3]
    #统计30天 放量  【1.2， 1.6】    
    return res
    

## get_peak_valley_stats

In [2]:
def get_peak_vally_stats(base,zig):
    kline=base.copy()

    kline['zig'] = peak_valley_pivots(kline.close, zig, -zig)
#     获取peak close
    peak_temp = (kline.zig == 1) *  kline.close
    peak_temp.replace(0,np.nan,inplace=True)
    kline['peak'] = peak_temp.ffill()

#     获取valley close
    valley_temp = (kline.zig == -1) *  kline.close
    valley_temp.replace(0,np.nan,inplace=True)
    kline['valley'] = valley_temp.ffill()

    # 将peak valley 整合到一列
    p_v_temp = (kline.zig != 0) * kline.close
    p_v_temp = (kline.zig != 0) * kline.close
    p_v_temp.replace(0,np.nan,inplace=True)
    kline['p_v_value'] = p_v_temp.ffill()
    
    #     backword
    kline['p_v_value_bw'] = p_v_temp.bfill()
    kline['p_pos'] = barlast(kline.zig == 1)
    kline['v_pos'] = barlast(kline.zig == -1)
    # 将v_pos p_pos整合到一列
    kline['z_pos'] = barlast(kline.zig != 0)
    
#     当前上涨，算前面一个下跌波段
    kline['dw_pct_last']  =  round((kline.p_v_value/kline.peak - 1)*1000,4)
#     当前下跌，算前面一个上涨波段
    kline['up_pct_last'] = round((kline.p_v_value/kline.valley - 1)*1000,4)

    # 上个波段的涨跌幅
    kline['z_pct_prev'] = kline[['dw_pct_last','up_pct_last']].sum(axis=1)

    #     需要对单k的zig 进行处理
    z_pos_raw = kline.z_pos + 1
    z_pos_raw_prev= z_pos_raw.shift(1)
    z_len_prev_temp = (z_pos_raw == 1) * z_pos_raw_prev 
    z_len_prev_temp.replace(0,np.nan,inplace=True)
    kline['z_len_prev_raw'] = z_len_prev_temp.ffill()
    kline['z_len_prev'] = kline.z_len_prev_raw

    up_pct_last_raw = kline.up_pct_last.dropna()
    dw_pct_last_raw = kline.dw_pct_last.dropna()
    up_pct_last_val = up_pct_last_raw[up_pct_last_raw != 0]
    dw_pct_last_val = dw_pct_last_raw[dw_pct_last_raw != 0]
    up_uniq = set(up_pct_last_val) #set(kline.up_pct_last.dropna())
#     对多个上涨波段涨跌幅的评价
    kline['up_pct_avg'] = sum(up_uniq)/(len(up_uniq) - 1) # -1 把0 扣掉

    dw_uniq = set(dw_pct_last_val) #set(kline.dw_pct_last.dropna())
#     对多个下跌波段涨跌幅的评价    
    kline['dw_pct_avg'] = sum(dw_uniq)/(len(dw_uniq)-1) # -1 把0 扣掉
    kline['p_v_value_prev'] = kline.p_v_value.shift(1)

    # no peak and valley
    non_zig_idx = (kline.z_pos != 0)
    v_pos_idx = (kline.v_pos == 0)
    p_pos_idx = (kline.p_pos == 0)
    zig_pct_chg = (kline.close / kline.p_v_value - 1) * 1000
    kline.loc[non_zig_idx,'z_pct_curr'] = zig_pct_chg[non_zig_idx]
    kline.loc[non_zig_idx,'z_pct_curr'] = zig_pct_chg[non_zig_idx]

    z_pos_pct_value = (kline.close / kline.p_v_value_prev - 1) *1000
    kline.loc[v_pos_idx,'z_pct_curr'] = z_pos_pct_value[v_pos_idx]
    kline.loc[p_pos_idx,'z_pct_curr'] = z_pos_pct_value[p_pos_idx]

    up_idx = (kline.v_pos == kline.z_pos)
    dw_idx = (kline.p_pos == kline.z_pos)
    kline.loc[up_idx,'z_pct_avg'] = kline.loc[up_idx,'up_pct_avg']
    kline.loc[dw_idx,'z_pct_avg'] = kline.loc[dw_idx,'dw_pct_avg']
#     zig_pct_avg_k = (sum(up_uniq) + abs(sum(dw_uniq))) / (len(up_uniq) + len(dw_uniq) - 2)
    zig_pct_avg_k = (sum(up_uniq) + abs(sum(dw_uniq))) /(up_pct_last_val.shape[0] + dw_pct_last_val.shape[0])
    kline['up_pct_avg_k'] = sum(up_uniq)/up_pct_last_val.shape[0]  # 获得上涨区间平均K的涨幅
    kline['dw_pct_avg_k'] = sum(dw_uniq)/dw_pct_last_val.shape[0]  # 获得下跌区间平均K的跌幅
    kline['z_len_curr'] = kline.z_pos.shift(1) + 1
    kline['z_len_full'] = (kline.z_pos==0) * kline.z_len_curr
    kline.z_len_full.replace(0,np.nan,inplace=True)
    kline.z_len_full.bfill(inplace=True)
    kline['z_pct_curr_avg_k'] = kline.z_pct_curr / kline.z_len_curr
    kline['z_pct_prev_avg_k'] = kline.z_pct_prev / kline.z_len_prev
    kline['z_pct_avg_k'] = zig_pct_avg_k
    kline.ffill(inplace=True)
  
    kline['p_prev1_'] = get_nth_peak(peak_temp,1).ffill() # 前一个peak
    kline['v_prev1_'] = get_nth_valley(valley_temp,1).ffill()  #前一个valley
#     kline['v_pos_prev1_'] = get_nth_peak(kline.)
#     下跌时可以算上涨侧的涨幅 p_pos = z_pos, peak - valley_prev1 时定值， v_pos - p_pos 是定值
    dw_side = kline.p_pos == kline.z_pos
    kline['up_pct_temp'] = (kline.peak / kline.valley - 1) *1000
    kline['up_cnt_'] = kline.v_pos - kline.p_pos
    kline['up_pct_avg_temp'] = kline.up_pct_temp / kline.up_cnt_
    kline.loc[dw_side,'up_pct_'] = kline.loc[dw_side,'up_pct_temp']
    kline.loc[dw_side,'up_pct_avg_'] = kline.loc[dw_side,'up_pct_avg_temp']
    
#     上涨时可以算下跌侧的涨幅 v_pos = z_pos, peak - valley_prev1 时定值， v_pos - p_pos 是定值
    up_side = kline.v_pos == kline.z_pos
    kline['dw_pct_temp'] = ( kline.valley/kline.peak - 1) *1000
    kline['dw_cnt_'] = (kline.p_pos - kline.v_pos)
    kline['dw_pct_avg_temp'] = kline.dw_pct_temp / kline.dw_cnt_
    kline.loc[up_side,'dw_pct_'] = kline.loc[up_side,'dw_pct_temp']
    kline.loc[up_side,'dw_pct_avg_'] = kline.loc[up_side,'dw_pct_avg_temp']
   
    kline.drop(columns=['zig','z_len_prev_raw','p_v_value_prev','up_pct_temp','up_pct_avg_temp','dw_pct_temp','dw_pct_avg_temp'], inplace=True)
    return kline


In [21]:
def get_peak_vally_stats1(base,zig):
    kline=base.copy()

    kline['zig'] = peak_valley_pivots(kline.close, zig, -zig)
#     获取peak close
    peak_temp = (kline.zig == 1) *  kline.close
    peak_temp.replace(0,np.nan,inplace=True)
    kline['peak'] = peak_temp.ffill()

#     获取valley close
    valley_temp = (kline.zig == -1) *  kline.close
    valley_temp.replace(0,np.nan,inplace=True)
    kline['valley'] = valley_temp.ffill()

    # 将peak valley 整合到一列
    p_v_temp = (kline.zig != 0) * kline.close
    p_v_temp = (kline.zig != 0) * kline.close
    p_v_temp.replace(0,np.nan,inplace=True)
    kline['p_v_value'] = p_v_temp.ffill()
    
    #     backword
    kline['p_v_value_bw'] = p_v_temp.bfill()
    kline['p_pos'] = barlast(kline.zig == 1)
    kline['v_pos'] = barlast(kline.zig == -1)
    # 将v_pos p_pos整合到一列
    kline['z_pos'] = barlast(kline.zig != 0)
    
    kline['dw_pct_last']  =  round((kline.p_v_value/kline.peak - 1)*1000,4)
    kline['up_pct_last'] = round((kline.p_v_value/kline.valley - 1)*1000,4)

    # 整合dw_pct_last up_pct_last
    kline['z_pct_prev'] = kline[['dw_pct_last','up_pct_last']].sum(axis=1)

    #     需要对单k的zig 进行处理
    z_pos_raw = kline.z_pos + 1
    z_pos_raw_prev= z_pos_raw.shift(1)
    z_len_prev_temp = (z_pos_raw == 1) * z_pos_raw_prev 
    z_len_prev_temp.replace(0,np.nan,inplace=True)
    kline['z_len_prev_raw'] = z_len_prev_temp.ffill()
    kline['z_len_prev'] = kline.z_len_prev_raw

    up_pct_last_raw = kline.up_pct_last.dropna()
    dw_pct_last_raw = kline.dw_pct_last.dropna()
    up_pct_last_val = up_pct_last_raw[up_pct_last_raw != 0]
    dw_pct_last_val = dw_pct_last_raw[dw_pct_last_raw != 0]
    up_uniq = set(up_pct_last_val) #set(kline.up_pct_last.dropna())
    kline['up_pct_avg'] = sum(up_uniq)/(len(up_uniq) - 1) # -1 把0 扣掉

    dw_uniq = set(dw_pct_last_val) #set(kline.dw_pct_last.dropna())
#     计算avg下跌波段涨跌幅
    kline['dw_pct_avg'] = sum(dw_uniq)/(len(dw_uniq)-1) # -1 把0 扣掉
    kline['p_v_value_prev'] = kline.p_v_value.shift(1)

    # no peak and valley
    non_zig_idx = (kline.z_pos != 0)
    v_pos_idx = (kline.v_pos == 0)
    p_pos_idx = (kline.p_pos == 0)
    zig_pct_chg = (kline.close / kline.p_v_value - 1) * 1000
    kline.loc[non_zig_idx,'z_pct_curr'] = zig_pct_chg[non_zig_idx]
    kline.loc[non_zig_idx,'z_pct_curr'] = zig_pct_chg[non_zig_idx]
# z_pct_curr 是当前波段的涨跌幅
    z_pos_pct_value = (kline.close / kline.p_v_value_prev - 1) *1000
    kline.loc[v_pos_idx,'z_pct_curr'] = z_pos_pct_value[v_pos_idx]
    kline.loc[p_pos_idx,'z_pct_curr'] = z_pos_pct_value[p_pos_idx]

    up_idx = (kline.v_pos == kline.z_pos)
    dw_idx = (kline.p_pos == kline.z_pos)
    kline.loc[up_idx,'z_pct_avg'] = kline.loc[up_idx,'up_pct_avg']
    kline.loc[dw_idx,'z_pct_avg'] = kline.loc[dw_idx,'dw_pct_avg']
#     zig_pct_avg_k = (sum(up_uniq) + abs(sum(dw_uniq))) / (len(up_uniq) + len(dw_uniq) - 2)
    zig_pct_avg_k = (sum(up_uniq) + abs(sum(dw_uniq))) /(up_pct_last_val.shape[0] + dw_pct_last_val.shape[0])
    kline['up_pct_avg_k'] = sum(up_uniq)/up_pct_last_val.shape[0]  # 获得上涨区间平均K的涨幅
    kline['dw_pct_avg_k'] = sum(dw_uniq)/dw_pct_last_val.shape[0]  # 获得下跌区间平均K的跌幅
    kline['z_len_curr'] = kline.z_pos.shift(1) + 1
    kline['z_len_full'] = (kline.z_pos==0) * kline.z_len_curr # 求波段周期数
    kline.z_len_full.replace(0,np.nan,inplace=True)
    kline.z_len_full.bfill(inplace=True) #往后填充波段周期数
    kline['z_pct_curr_avg_k'] = kline.z_pct_curr / kline.z_len_curr
    kline['z_pct_prev_avg_k'] = kline.z_pct_prev / kline.z_len_prev
    kline['z_pct_avg_k'] = zig_pct_avg_k  # 平均总体波段的平均单K涨跌幅
    kline.ffill(inplace=True)
 
    # pct chg since last valley
    kline['valley_ref1'] = kline.valley.shift(1)
    non_v_idx = kline.v_pos != 0 
    v_pct_curr = (kline.close / kline.valley - 1) * 1000
    kline.loc[non_v_idx,'v_pct_curr'] = v_pct_curr[non_v_idx]
    v_idx = (kline.v_pos == 0) & (kline.v_pos.shift(1) != 1)
    v_pct_curr =(kline.close / kline.valley_ref1 - 1) * 1000
    kline.loc[v_idx,'v_pct_curr'] = v_pct_curr[v_idx]

    # pct chg since last peak
    kline['peak_ref1'] = kline.peak.shift(1)
    non_p_idx = kline.p_pos != 0
    p_pct_curr = (kline.close / kline.peak - 1) * 1000
    kline.loc[non_p_idx,'p_pct_curr'] = p_pct_curr[non_p_idx]
    p_idx = kline.p_pos == 0
    p_pct_curr =(kline.close / kline.peak_ref1 - 1) * 1000
    kline.loc[p_idx,'p_pct_curr'] = p_pct_curr[p_idx]

    valley_last_temp = (kline.v_pos == 0) *  kline.valley_ref1
    valley_last_temp.replace(0,np.nan,inplace=True)
    kline['valley_last'] = valley_last_temp.ffill()

    peak_last_temp = (kline.p_pos == 0) *  kline.peak_ref1
    peak_last_temp.replace(0,np.nan,inplace=True)
    kline['peak_last'] = peak_last_temp.ffill()

    kline['v_pct_bw'] = (kline.p_v_value_bw / kline.valley - 1) * 1000
    kline['p_pct_bw'] = (kline.p_v_value_bw / kline.peak - 1) *1000
    dw_side = kline.z_pos == kline.v_pos
    kline.loc[dw_side,'z_pct_curr_full'] = kline.v_pct_bw 
    up_side = kline.z_pos == kline.p_pos
    kline.loc[up_side,'z_pct_curr_full'] = kline.p_pct_bw 
    kline.z_pct_curr_full.replace(0,np.nan,inplace=True)
    kline.z_pct_curr_full.bfill(inplace=True)
    # z_pos连续0
    cond_2_z_pos = (kline.z_pos.shift(-1) == 0) & (kline.z_pos ==0)
#     kline['z_pct_curr_next'] = kline.z_pct_curr.shift(-1)
#     kline.loc[cond_2_z_pos,'z_pct_curr_full'] = kline.loc[cond_2_z_pos,'z_pct_curr_next']  
    kline.loc[cond_2_z_pos,'z_pct_curr_full'] = kline.loc[cond_2_z_pos,'z_pct_curr']  
    
#     建立zig index，方便后面分析
    kline['zig_flag'] = 0
    zig_pos = (kline.z_pos == 0)
    kline.loc[zig_pos,'zig_flag'] = 1

    #     建立zig index，方便后面分析
    kline['zig_flag'] = 0
    zig_pos = (kline.z_pos == 0)
    kline.loc[zig_pos,'zig_flag'] = 1
    kline['zig_index'] =kline.zig_flag.cumsum()
   
    kline['p_prev1_'] = get_nth_peak(peak_temp,1).ffill() # 前一个peak
    kline['v_prev1_'] = get_nth_valley(valley_temp,1).ffill()  #前一个valley
#     kline['v_pos_prev1_'] = get_nth_peak(kline.)
    
    
#     波段, 上一个同方向波段的涨跌幅
    kline['bd_zdf_curr_'] = (kline.peak/kline.valley - 1)*100
    kline['bd_zdf_prev1_'] = (kline.p_prev1_/kline.v_prev1_ - 1)*100
    kline['bd_zdf_prev1_'] = (kline.peak/kline.valley - 1)*100 / (kline.v_pos - kline.v_pos)
    kline['bd_zdf_prev2_'] = (kline.p_prev1_/kline.v_prev1_ - 1)*100    
    
    kline.drop(columns=['zig','z_len_prev_raw','p_v_value_prev','peak_ref1','valley_ref1'], inplace=True)
    return kline

#  series拼接，把series变字典，然后concat   
#     df = {
#         'zig': klines.zig,
#         'p_pos': klines.p_pos,
#         'v_pos': klines.v_pos,
#         'peak': peak,
#         'valley': valley,
#         'p_v_value': p_v_value
#     } 
#     return pd.concat(df,axis=1)  

## get_nearest_supp_resis

In [124]:
def get_nearest_supp_resis(price,supp_resis):
    price_arr = price[['close']].values
    supp_resis_price_diff = (supp_resis.values / price_arr - 1) * 1000
    pos = np.abs(supp_resis_price_diff).argmin(axis=1)
#     nearest_supp_resis = supp_resis_price_diff[np.arange(price_arr.shape[0]), pos]
    return supp_resis_price_diff[np.arange(price_arr.shape[0]), pos]

## detect_yd

发现异动 zig=0.3/100  千分之3  
1 出现快速下跌  
2 出现快速上涨  

## get_bsp

In [11]:
def get_bsp(data,bsp_n,body_thresh,bsp_thresh1,bsp_thresh2):
    kline=data[['close','open','volume']].copy()
    body = kline.close - kline.open
    mab = ma(abs(body),bsp_n)  # ma body
    # # buy sell pressure body_thresh=0.3 
    need_handle = abs(body) < mab*body_thresh
    keep_body = abs(body) >= mab*body_thresh
#     对小K线或者十字星 的body 长度进行 标准化
    kline.loc[need_handle,'revised_body'] = mab[need_handle]*body_thresh
    kline.loc[keep_body,'revised_body'] = abs(body[keep_body])
    bsp=kline.volume/kline.revised_body
    ma_bsp = ma(bsp,bsp_n)
#     kline['ma_bsp'] =ma_bsp
    bsp_ratio = round(bsp/ma_bsp,4)
    kline['bsp_ratio'] = bsp_ratio
#     超轻买卖压  代表大家在观望
    light_idx = bsp_ratio < bsp_thresh1
    kline.loc[light_idx,'bsp_label'] = 1
# 超重买卖压  代表分歧，可能就是一个阶段的临界值
    heavey_idx = (bsp_thresh2 <= bsp_ratio)
    kline.loc[heavey_idx,'bsp_label'] = 3
# 普通买卖压
    normal_idx = (bsp_thresh1 <= bsp_ratio) & (bsp_ratio < bsp_thresh2)
    kline.loc[normal_idx,'bsp_label'] = 2
#     计算相对bsp 
    kline['bsp_relative'] = bsp_ratio/bsp_ratio.shift(1)
#     kline=kline.fillna(2)
    return kline[['bsp_label','bsp_ratio','bsp_relative']]
#     return kline[['bsp_label','revised_body','bsp_ratio']]

In [15]:
def get_bsp2(data,bsp_n,body_thresh,bsp_thresh1,bsp_thresh2):
    kline=data[['time','close','open','volume']].copy()
    body = kline.close - kline.open
    mab = ma(abs(body),bsp_n)  # ma body
    # # buy sell pressure body_thresh=0.3 
    need_handle = abs(body) < mab*body_thresh
    keep_body = abs(body) >= mab*body_thresh
#     对小K线或者十字星 的body 长度进行 标准化
    kline.loc[need_handle,'revised_body'] = mab[need_handle]*body_thresh
    kline.loc[keep_body,'revised_body'] = body[keep_body]
    bsp=kline.volume/kline.revised_body
    ma_bsp = ma(abs(bsp),bsp_n)
    kline['ma_bsp'] =ma_bsp
    kline['mab'] = mab
    kline['bsp'] = bsp
    kline['body'] = body
    kline['keep_mab'] = keep_mab
    
    bsp_ratio = round(bsp/ma_bsp,4)
    kline['bsp_ratio'] = bsp_ratio
#     超轻买卖压  代表大家在观望
    light_idx = bsp_ratio < bsp_thresh1
    kline.loc[light_idx,'bsp_label'] = 1
# 超重买卖压  代表分歧，可能就是一个阶段的临界值
    heavey_idx = (bsp_thresh2 <= bsp_ratio)
    kline.loc[heavey_idx,'bsp_label'] = 3
# 普通买卖压
    normal_idx = (bsp_thresh1 <= bsp_ratio) & (bsp_ratio < bsp_thresh2)
    kline.loc[normal_idx,'bsp_label'] = 2
#     kline=kline.fillna(2)
#     return kline[['bsp_label','bsp_ratio']]
    return kline[['time','body','revised_body','volume','bsp_label','bsp_ratio','ma_bsp','mab','bsp']]
#     return kline[['bsp_label','revised_body','bsp_ratio']]

## handle_opening

In [112]:
def handle_opening(base):
    kline = base.copy()
    kline['hhmm'] = kline.time.str.split('-',n=1,expand=True)
    opening = kline.hhmm=='09:05'
    

## gen_stop_loss_long

In [19]:
# para = {'K线':{
#             '大阴': 2,
#             '大阳': 2
#             },
#         'bsp':{
#             'relative_yc':3 #bsp/bsp.shift(1)
#             },
#         'volume': {
#             'fl_v1_strength': 1
#             },
#         'stop_loss': {
#             'long': {'buffer_ratio': 2}, # 用于乘ma_body 用于buffer
#             'short': {}
#             }
#        }

## gen_short_stop_loss

# 计算

In [1]:
# 对均线的短中长周期设参数
# s1,s2,s3 = 4, 8, 24
# s1,s2,s3 = 8, 16, 48
# s1,s2,s3 = 12, 24, 72
# s1,s2,s3 = 20,40,120
# s0=3
# s1,s2,s3 = 8,16,32
# m1,m2,m3 = 32,64,128
# l1,l2,l3 = 128,256,512
# s1,s2,s3 = 8,13,21
# m1,m2,m3 = 21,34,55
# l1,l2,l3 = 55,89,144
# 8,16,32,64,128,256,512

# s1,s2,s3 = 6,12,24
# m1,m2,m3 = 24,48,96
# l1,l2,l3 = 96,192,384


kls=0.2
klm=1
kls=1.2
zig1=0.2/100
zig2=0.3/100
zig3=0.5/100
zig4=0.8/100
zig5=1.3/100
zig6=2.1/100
yc_vol_ratio=2 # 异常vol ratio

# 对长上影/下影分类  ma_body 的2倍以上就是
ups,lows = 2,2
#回踩反抽
huicai_range = 0.5
# 对vol 最高第二高进行定位
vs=10  #13量的变化更快，采用更短的周期
vm=40  # 46
vl=200  #200
vThresh = 0.15

# 用于计算买卖压的
body_thresh=0.3
bsp_thresh1 = 1.4
bsp_thresh2 = 2.1
bsp_n = 20

# 计算n周期ma_body
n_ma_body=20
# 和高点比较（千分之）
peak_var_pct = 1.5 
cs=20 
cm=46
cl=112 #1 Day

# 靠拢的参数: ma_body的倍数
# ma1-3
kl_s_ratio = 1
# ma3-5
kl_m_ratio = 1.5

## calc_base_data

In [1]:
def calc_base_data(klines):
    klinesC = klines.copy()
    
    klinesC['hhmm'] = klinesC.time.str.split('-',n=1,expand=True).loc[:,1]
    klinesC['opening_day'] = klinesC.hhmm=='09:05'
    klinesC['end_day'] = klinesC.hhmm == '15:00'
    klinesC['opening_night'] = klinesC.hhmm=='21.05'
    klinesC['end_day'] = klinesC.hhmm == '23:00'
    klinesC['vol_prev'] = klinesC.volume.shift(1)
    klinesC['c_prev'] = klinesC.close.shift(1)
    
    # 基础数据 均线类
    klinesC['ma0'] = MA(klinesC, conf['ma']['m0'])
    klinesC[['ma1','ma2','ma3','ma4','ma5','ma6','ma7']] = get_ema(klinesC,conf)
    
    klinesC['ma12_ratio'] = klinesC.ma1/klinesC.ma2
    klinesC['ma23_ratio'] = klinesC.ma2/klinesC.ma3
    klinesC['ma34_ratio'] = klinesC.ma3/klinesC.ma4
    klinesC['ma45_ratio'] = klinesC.ma4/klinesC.ma5
    klinesC['ma56_ratio'] = klinesC.ma5/klinesC.ma6
    klinesC['ma67_ratio'] = klinesC.ma6/klinesC.ma7
    klinesC['ma13_ratio'] = klinesC.ma1/klinesC.ma3
    klinesC['ma24_ratio'] = klinesC.ma2/klinesC.ma4
    klinesC['ma35_ratio'] = klinesC.ma3/klinesC.ma5
    klinesC['ma46_ratio'] = klinesC.ma4/klinesC.ma6
    klinesC['ma57_ratio'] = klinesC.ma5/klinesC.ma7
    
    # get vol stats
    # 基础数据 均量类
#     klinesC['mv1'] = ma(klinesC.volume, vs)
#     klinesC['mv2'] = ma(klinesC.volume, vm)
#     klinesC['mv3'] = ma(klinesC.volume, vl)
    klinesC['mv1'] = klinesC.volume.ewm(span=vs,adjust=False).mean()
    klinesC['mv2'] = klinesC.volume.ewm(span=vm,adjust=False).mean()
    klinesC['mv3'] = klinesC.volume.ewm(span=vl,adjust=False).mean()

    # 衍生基础数据 斜率
    klinesC['slp0'] = round(klinesC.ma0 / klinesC.ma0.shift(1) * 1000, 4)
    klinesC['slp1'] = round(klinesC.ma1 / klinesC.ma1.shift(1) * 1000, 4)
    klinesC['slp2'] = round(klinesC.ma2 / klinesC.ma2.shift(1) * 1000, 4)
    klinesC['slp3'] = round(klinesC.ma3 / klinesC.ma3.shift(1) * 1000, 4)
    klinesC['slp4'] = round(klinesC.ma4 / klinesC.ma4.shift(1) * 1000, 4)
    klinesC['slp5'] = round(klinesC.ma5 / klinesC.ma5.shift(1) * 1000, 4)
    klinesC['slp6'] = round(klinesC.ma6 / klinesC.ma6.shift(1) * 1000, 4)
    klinesC['slp7'] = round(klinesC.ma7 / klinesC.ma7.shift(1) * 1000, 4)
  
 
    # 衍生基础数据 偏离 
#     klinesC['bias0'] = round((klinesC.close / klinesC.ma0 - 1) * 1000, 2)
#     klinesC['bias1'] = round((klinesC.close / klinesC.ma1 - 1) * 1000, 2)
#     klinesC['bias2'] = round((klinesC.close / klinesC.ma2 - 1) * 1000, 2)
#     klinesC['bias3'] = round((klinesC.close / klinesC.ma3 - 1) * 1000, 2) 
#     klinesC['bias4'] = round((klinesC.close / klinesC.ma4 - 1) * 1000, 2)
#     klinesC['bias5'] = round((klinesC.close / klinesC.ma5 - 1) * 1000, 2)
#     klinesC['bias6'] = round((klinesC.close / klinesC.ma6 - 1) * 1000, 2)
#     klinesC['bias7'] = round((klinesC.close / klinesC.ma7 - 1) * 1000, 2)
    
    klinesC['bias1'] = round((1 - klinesC.close / klinesC.ma1.shift(1)) * 1000, 4)
    klinesC['bias2'] = round((1 - klinesC.close / klinesC.ma2.shift(1)) * 1000, 4)
    klinesC['bias3'] = round((1 - klinesC.close / klinesC.ma3.shift(1)) * 1000, 4) 
    klinesC['bias4'] = round((1 - klinesC.close / klinesC.ma4.shift(1)) * 1000, 4)
    klinesC['bias5'] = round((1 - klinesC.close / klinesC.ma5.shift(1)) * 1000, 4)
    klinesC['bias6'] = round((1 - klinesC.close / klinesC.ma6.shift(1)) * 1000, 4)
    klinesC['bias7'] = round((1 - klinesC.close / klinesC.ma7.shift(1)) * 1000, 4)
 
    #均线衍生 均线间距
    klinesC['df_ma01'] = klinesC.ma0 - klinesC.ma1
    klinesC['df_ma12'] = klinesC.ma1 - klinesC.ma2
    klinesC['df_ma23'] = klinesC.ma2 - klinesC.ma3
    klinesC['df_ma34'] = klinesC.ma3 - klinesC.ma4
    klinesC['df_ma45'] = klinesC.ma4 - klinesC.ma5  
    klinesC['df_ma56'] = klinesC.ma5 - klinesC.ma6    
    klinesC['df_ma67'] = klinesC.ma6 - klinesC.ma7
    
#     klinesC['df_ma12'] = round((klinesC.ma1 - klinesC.ma2)*1000/klinesC.close,4)
#     klinesC['df_ma23'] = round((klinesC.ma2 - klinesC.ma3)*1000/klinesC.close,4)
#     klinesC['df_ma13'] = round((klinesC.ma1 - klinesC.ma3)*1000/klinesC.close,4)
#     klinesC['df_ma34'] = round((klinesC.ma3 - klinesC.ma4)*1000/klinesC.close,4)
#     klinesC['df_ma24'] = round((klinesC.ma2 - klinesC.ma4)*1000/klinesC.close,4)
#     klinesC['df_ma45'] = round((klinesC.ma4 - klinesC.ma5)*1000/klinesC.close,4)
#     klinesC['df_ma35'] = round((klinesC.ma3 - klinesC.ma5)*1000/klinesC.close,4)
#     klinesC['df_ma56'] = round((klinesC.ma5 - klinesC.ma6)*1000/klinesC.close,4)
#     klinesC['df_ma46'] = round((klinesC.ma4 - klinesC.ma6)*1000/klinesC.close,4)
#     klinesC['df_ma67'] = round((klinesC.ma6 - klinesC.ma7)*1000/klinesC.close,4)
#     klinesC['df_ma57'] = round((klinesC.ma5 - klinesC.ma7)*1000/klinesC.close,4)

    # 衍生数据 偏离的最大最小值
    klinesC['ma_max_13'] = klinesC[['ma1','ma2','ma3']].max(1)
    klinesC['ma_min_13'] = klinesC[['ma1','ma2','ma3']].min(1)
    
#     klinesC['ma_max_14'] = klinesC[['ma1','ma2','ma3','ma4','ma5']].max(1)
#     klinesC['ma_min_14'] = klinesC[['ma1','ma2','ma3','ma4','ma5']].min(1)
    klinesC['ma_max_14'] = klinesC[['ma1','ma2','ma3','ma4']].max(1)
    klinesC['ma_min_14'] = klinesC[['ma1','ma2','ma3','ma4']].min(1)    
    
    klinesC['ma_max_24'] = klinesC[['ma2','ma3','ma4']].max(1)
    klinesC['ma_min_24'] = klinesC[['ma2','ma3','ma4']].min(1)
    
    klinesC['ma_max_35'] = klinesC[['ma3','ma4','ma5']].max(1)
    klinesC['ma_min_35'] = klinesC[['ma3','ma4','ma5']].min(1)
    
    klinesC['ma_max_46'] = klinesC[['ma4','ma5','ma6']].max(1)
    klinesC['ma_min_46'] = klinesC[['ma4','ma5','ma6']].min(1)
    
    klinesC['ma_max_57'] = klinesC[['ma5','ma6','ma7']].max(1)
    klinesC['ma_min_57'] = klinesC[['ma5','ma6','ma7']].min(1)
    
    klinesC['ma_max_15'] = klinesC[['ma1','ma2','ma3','ma4','ma5']].max(1)
    klinesC['ma_min_15'] = klinesC[['ma1','ma2','ma3','ma4','ma5']].min(1) 
 
    klinesC['ma_max_17'] = klinesC[['ma1','ma2','ma3','ma4','ma5','ma6','ma7']].max(1)
    klinesC['ma_min_17'] = klinesC[['ma1','ma2','ma3','ma4','ma5','ma6','ma7']].min(1) 
    
    # 3ma 间隔
    klinesC['rng_ma13'] = klinesC.ma_max_13 - klinesC.ma_min_13
    klinesC['rng_ma24'] = klinesC.ma_max_24 - klinesC.ma_min_24
    klinesC['rng_ma35'] = klinesC.ma_max_35 - klinesC.ma_min_35
    klinesC['rng_ma46'] = klinesC.ma_max_46 - klinesC.ma_min_46
    klinesC['rng_ma57'] = klinesC.ma_max_57 - klinesC.ma_min_57

    klinesC['rng_ma14'] = klinesC.ma_max_14 - klinesC.ma_min_14
    klinesC['rng_ma15'] = klinesC.ma_max_15 - klinesC.ma_min_15
    klinesC['rng_ma17'] = klinesC.ma_max_17 - klinesC.ma_min_17
            
    # 均线衍生 平均斜率
    klinesC['mslp0'] = ma(klinesC.slp0,3)
    klinesC['mslp1'] = ma(klinesC.slp1, 3) # 用于判断回调
    klinesC['mslp2'] = ma(klinesC.slp2, 6)
    klinesC['mslp3'] = ma(klinesC.slp3, 12)
    klinesC['mslp4'] = ma(klinesC.slp4, 24)
    klinesC['mslp5'] = ma(klinesC.slp5, 36)
    klinesC['mslp6'] = ma(klinesC.slp6, 48)
    klinesC['mslp7'] = ma(klinesC.slp7, 60)

    #均线衍生 均绝对偏离
    #     klinesC['ma_2_slp0'] = ma(klinesC.slp0, 2)     klinesC['ma_body'] = ma(abs(klinesC.cdlBody), 20)
#     klinesC['mbias1s'] = ma(abs(klinesC.bias1s), 20)  # 获取20日绝对bias1的平均
#     klinesC['mbias2s'] = ma(abs(klinesC.bias2s), 20)  # 获取20日绝对bias2的平均
#     klinesC['mbias3s'] = ma(abs(klinesC.bias3s), 20) 
    
#     klinesC['mbias1m'] = ma(abs(klinesC.bias1m), 30)  # 获取20日绝对bias1的平均
#     klinesC['mbias2m'] = ma(abs(klinesC.bias2m), 30)  # 获取20日绝对bias2的平均
#     klinesC['mbias3m'] = ma(abs(klinesC.bias3m), 30) 
    
#     klinesC['mbias1l'] = ma(abs(klinesC.bias1l), 40)  # 获取20日绝对bias1的平均
#     klinesC['mbias2l'] = ma(abs(klinesC.bias2l), 40)  # 获取20日绝对bias2的平均
#     klinesC['mbias3l'] = ma(abs(klinesC.bias3l), 4 0) 

    #均线衍生 均偏离    
    klinesC['m_bias1'] = ma(klinesC.bias1,7)
    klinesC['m_bias2'] = ma(klinesC.bias2,10)
    klinesC['m_bias3'] = ma(klinesC.bias3,17)
    klinesC['m_bias4'] = ma(klinesC.bias4,20)
    klinesC['m_bias5'] = ma(klinesC.bias5,25)
    klinesC['m_bias6'] = ma(klinesC.bias6,30)
    klinesC['m_bias7'] = ma(klinesC.bias7,40)
    
#     klinesC['m_bias8'] = ma(klinesC.bias8,30)
#     klinesC['m_bias9'] = ma(klinesC.bias9,40)    
    
#   均线衍生  计算价格对平均偏移的diff
#     klinesC['m1_bias'] = klinesC.close - klinesC.m_bias1
#     klinesC['m1_bias'] = klinesC.close - klinesC.m_bias2
#     klinesC['m1_bias'] = klinesC.close - klinesC.m_bias3
#     klinesC['m1_bias'] = klinesC.close - klinesC.m_bias4
#     klinesC['m1_bias'] = klinesC.close - klinesC.m_bias5
#     klinesC['m1_bias'] = klinesC.close - klinesC.m_bias6
#     klinesC['m1_bias'] = klinesC.close - klinesC.m_bias7 
       
    # 基础 计算K线的特征
    klinesC['pct_chg'] = round(klinesC.close.pct_change() * 1000,3)
    klinesC['zdf'] = klinesC.close - klinesC.close.shift(1)
    klinesC['abs_zdf'] = abs(klinesC.zdf)
    klinesC['kbody'] = klinesC.close - klinesC.open
    klinesC['abs_body'] = abs(klinesC.kbody)
    klinesC['ma_body'] = ma(klinesC.abs_body,n_ma_body)
    klinesC['bd_ratio'] = klinesC.kbody/klinesC.ma_body
    klinesC['kbody_top'] = klinesC[['close','open']].max(axis=1)
    klinesC['kbody_button'] = klinesC[['close','open']].min(axis=1)
    klinesC['kbody_pct'] = round(klinesC.kbody / klinesC.c_prev * 1000, 4) #k线实体的振幅
    klinesC['k_hl'] = klinesC.high - klinesC.low  # k线的 full length = H - L
    klinesC['k_hl_pct'] = round(klinesC.k_hl / klinesC.c_prev * 1000, 4)  # K线的振幅
    klinesC['k_ho'] = klinesC.high - klinesC.open  # 可以作为 多头
    klinesC['k_hc'] = klinesC.high - klinesC.close  # 可以作为空头
    klinesC['k_ups'] = klinesC[['k_ho', 'k_hc']].min(axis=1)  # 上影线upper shadow
    klinesC['k_ups_pct'] = round(klinesC.k_ups / klinesC.close * 1000, 4)  #上影线振幅
    klinesC['k_lo'] = klinesC.open - klinesC.low  # 可以作为 空头
    klinesC['k_lc'] = klinesC.close - klinesC.low  #
    klinesC['k_lows'] = klinesC[['k_lo', 'k_lc']].min(axis=1)  # 下影线 lower shadow
    klinesC['k_lows_pct'] = round(klinesC.k_lows / klinesC.close * 1000, 4) #下影线振幅
    klinesC['ma_kbody'] = ma(klinesC.abs_body, 20)
    klinesC['hc_ratio'] = klinesC.high / klinesC.close  # 用于判断高开阴线
    klinesC['lc_ratio'] = klinesC.low / klinesC.close # 用于判断低开阳线
    # 求均值    
    klinesC['ma_ups'] = ma(klinesC.k_ups, 20) # 获取upper shadow 均值
    klinesC['ma_lows'] = ma(klinesC.k_lows, 20) # 获取lower shadow 均值  
    klinesC['ma_pct'] = ma(abs(klinesC.pct_chg),20)
    
    # 计算买卖压
#     klinesC['zd_strength'] = klinesC.abs_body / klinesC.ma_kbody  # 涨跌力度
    klinesC['zd_strength'] = klinesC.zdf / klinesC.ma_kbody  # 涨跌力度
    klinesC['body_strength'] = klinesC.kbody / klinesC.ma_kbody  # 涨跌力度
    
    # 描述交易量
    # 交易量分类 vol strength
    klinesC['v1_strength'] = klinesC.volume / klinesC.mv1.shift(1)
    klinesC['v2_strength'] = klinesC.volume / klinesC.mv2.shift(1)
    klinesC['v3_strength'] = klinesC.volume / klinesC.mv3.shift(1)
    klinesC['mv13_ratio'] = klinesC.mv1 / klinesC.mv3
    klinesC['mv12_ratio'] = klinesC.mv1 / klinesC.mv2
    klinesC['mv23_ratio'] = klinesC.mv2 / klinesC.mv3
    
    # 巨量统计
    klinesC['jl_1d'] = (klinesC.v3_strength >= 3)
    klinesC['jl_tj'] = count(klinesC.jl_1d, 20)  # 巨量统计
    klinesC['jl_2d'] = klinesC.jl_1d & klinesC.jl_1d.shift(1)  # 连续2天放量
    klinesC['jl_3d'] = klinesC.jl_2d & klinesC.jl_2d.shift(1)  # 连续3天放量

#     #定位巨量位置
#     klinesC['jl_1d_last_days'] = barlast(klinesC.jl_1d == True)
#     klinesC['jl_2d_last_days'] = barlast(klinesC.jl_2d == True)
#     klinesC['jl_3d_last_days'] = barlast(klinesC.jl_3d == True)
    
    # 求最大量 第二/第三 量
#     klinesC['hhv_vol_'+ str(vs)] = hhv(klinesC.volume,vs)
    klinesC['hhv_vol_s'] = hhv(klinesC.volume,vs)
    klinesC['hhv_vol_m'] = hhv(klinesC.volume,vm)
    klinesC['hhv_vol_l'] = hhv(klinesC.volume,vl)
    
#     求最高close
    klinesC['hhv_close_s'] = hhv(klinesC.close,cs)
    klinesC['hhv_close_m'] = hhv(klinesC.close,cm)
    klinesC['hhv_close_l'] = hhv(klinesC.close,cl)
    
    #量价关系
    
    ## 形态数据 均线形态
#     klinesC['ktpl_13'] = ((klinesC.ma1 <= klinesC.ma2) & (klinesC.ma2 <= klinesC.ma3))  # 空头排列 ma1,2,3
#     klinesC['dtpl_13'] = ((klinesC.ma1 >= klinesC.ma2) & (klinesC.ma2 >= klinesC.ma3))  # 多头排列 ma1,2,3
#     klinesC['ktpl_46'] = ((klinesC.ma3 <= klinesC.ma4) & (klinesC.ma4 <= klinesC.ma6))  # 空头排列 ma1,2,3
#     klinesC['dtpl_46'] = ((klinesC.ma3 >= klinesC.ma4) & (klinesC.ma4 >= klinesC.ma6))  # 多头排列 ma1,2,3
#     klinesC['ktpl_79'] = ((klinesC.ma7 <= klinesC.ma8) & (klinesC.ma8 <= klinesC.ma9)) # 空头排列 ma2,3,4
#     klinesC['dtpl_79'] = ((klinesC.ma7 >= klinesC.ma8) & (klinesC.ma8 >= klinesC.ma9)) # 多头排列 ma2,3,4    
           
    # 价格zigzag

#  ['zig', 'peak', 'valley', 'p_v_value', 'p_pos', 'v_pos', 'z_pos','dw_pct_prev', 'up_pct_prev', 'z_pct_prev', 'z_len_prev','up_pct_avg',
# 'dw_pct_avg', 'z_pct_curr', 'z_pct_avg', 'up_pct_avg_k', 'dw_pct_avg_k','z_pct_curr_avg_k', 'z_pct_prev_avg_k', 'z_pct_avg_k']

    
#     klinesC['price_pos'] = get_price_pos(klinesC[['bias1','bias2','bias3','bias4','bias5','bias6','bias7','bias8','bias9']])
#     klinesC['min_bias'] = get_min_bias(klinesC[['bias1','bias2','bias3','bias4','bias5','bias6','bias7','bias8','bias9']])
    klinesC['min_bias13'] = get_min_bias(klinesC[['bias1','bias2','bias3']]) # dt_13 if min_bias13>0
    klinesC['min_bias35'] = get_min_bias(klinesC[['bias3','bias4','bias5']])
    klinesC['min_bias57'] = get_min_bias(klinesC[['bias5','bias6','bias7']])
    
#     bias=klinesC[['bias1','bias2','bias3','bias4','bias5','bias6','bias7','bias8','bias9']]
    bias=klinesC[['bias1','bias2','bias3','bias4','bias5','bias6','bias7']]
#     df_ma = klinesC[['df_ma12','df_ma23','df_ma34','df_ma45','df_ma56','df_ma67','df_ma78','df_ma89']]
    df_ma = klinesC[['df_ma12','df_ma23','df_ma34','df_ma45','df_ma56','df_ma67']]
    #price_pos min_bias 的输入为ma1-9
    #price_in_ma 0 - ma1-2, 1-> ma2-3, 3->ma3,ma5(第三第四ma)
    klinesC[['price_in_ma','price_in_ma_cnt','price_pos','min_bias']] = get_price_stage(bias,df_ma)
    klinesC['huicai'] = abs(klinesC.min_bias) <= huicai_range # 用于回踩或者反抽策略   
    
# 3均线多空形态
#     klinesC[['ma_stage1_3','ma_stage2_4','ma_stage3_5','ma_stage4_6','ma_stage5_7']]=get_ma_stage(bias)[0]
    # 3均线区间值
#     klinesC[['ma_rng1_3','ma_rng2_4','ma_rng3_5','ma_rng4_6','ma_rng5_7']]=get_ma_stage(bias)[1]
#     klinesC['price_stage'] = klinesC.apply()
#     klinesC['mv_stage'] =get_mv_stage(klinesC[['v1_strength','v2_strength','v3_strength','mv13_ratio','mv23_ratio','mv12_ratio']],1.2,1.5)[:,18]    
#     klinesC['ma_stage_24'] = get_3ma_stage(klinesC[['ma23_ratio','ma34_ratio','ma24_ratio']],1,3)[:,18]
#     klinesC['ma_stage_35'] = get_3ma_stage(klinesC[['ma34_ratio','ma45_ratio','ma35_ratio']],1,3)[:,18]
#     klinesC['ma_stage_46'] = get_3ma_stage(klinesC[['ma45_ratio','ma56_ratio','ma46_ratio']],1,3)[:,18]
#     klinesC['ma_stage_57'] = get_3ma_stage(klinesC[['ma56_ratio','ma67_ratio','ma57_ratio']],1,3)[:,18]

# 对均线排列进行分类
    klinesC['ma_category_s'] = get_3ma_category(klinesC[['ma12_ratio','ma23_ratio','ma13_ratio']])
    klinesC['ma_category_m'] = get_3ma_category(klinesC[['ma34_ratio','ma45_ratio','ma35_ratio']])
    klinesC['ma_category_l'] = get_3ma_category(klinesC[['ma56_ratio','ma67_ratio','ma57_ratio']])
# 对均线间距离进行分类
    klinesC[['kl_ma13','kl_ma35']] = get_3ma_rng_category(klinesC[['rng_ma13','rng_ma35','ma_body']],kl_s_ratio,kl_m_ratio)

# 把量的状态分为常量，放量，缩量 'cl_vol','sl_vol','fl_vol',
#     klinesC.loc[:,['mv_stage']] = get_mv_stage(klinesC[['mv12_ratio','mv23_ratio','mv13_ratio']],vThresh)
    klinesC['mv_stage'] = get_mv_stage(klinesC[['mv12_ratio','mv23_ratio','mv13_ratio']],vThresh)
#     klinesC['v_status'] = 

# 获取买卖压
#     klinesC['bsp'] = get_bsp(klinesC,40,0.3,0.7,2)  
    klinesC[['bsp_label','bsp_ratio','bsp_relative']] = get_bsp(klinesC,bsp_n,body_thresh,bsp_thresh1,bsp_thresh2)
    
#     klinesC['ma_category_13'] = get_3ma_category(klinesC[['ma12_ratio','ma23_ratio','ma13_ratio']])
#     klinesC['ma_category_24'] = get_3ma_category(klinesC[['ma23_ratio','ma34_ratio','ma24_ratio']])
#     klinesC['ma_category_35'] = get_3ma_category(klinesC[['ma34_ratio','ma45_ratio','ma35_ratio']])
#     klinesC['ma_category_46'] = get_3ma_category(klinesC[['ma45_ratio','ma56_ratio','ma46_ratio']])
#     klinesC['ma_category_57'] = get_3ma_category(klinesC[['ma56_ratio','ma67_ratio','ma57_ratio']])
    
    # 切入信号
    zig_res = get_peak_vally_stats(klinesC[['close']],conf['zig']['zig1'])
    zig_col = get_zig_col(zig_res,1)
    klinesC[zig_col] = zig_res

    zig_res = get_peak_vally_stats(klinesC[['close']],conf['zig']['zig2'])
    zig_col = get_zig_col(zig_res,2)
    klinesC[zig_col] = zig_res
    
    zig_res = get_peak_vally_stats(klinesC[['close']],conf['zig']['zig3'])
    zig_col = get_zig_col(zig_res,3)
    klinesC[zig_col] = zig_res
 
    zig_res = get_peak_vally_stats(klinesC[['close']],conf['zig']['zig4'])
    zig_col = get_zig_col(zig_res,4)
    klinesC[zig_col] = zig_res
    
    zig_res = get_peak_vally_stats(klinesC[['close']],conf['zig']['zig5'])
    zig_col = get_zig_col(zig_res,5)
    klinesC[zig_col] = zig_res

    zig_res = get_peak_vally_stats(klinesC[['close']],conf['zig']['zig6'])
    zig_col = get_zig_col(zig_res,6)
    klinesC[zig_col] = zig_res
    
    klinesC['peak1_bias'] = (klinesC.peak1 / klinesC.close - 1) * 1000
    klinesC['peak2_bias'] = (klinesC.peak2 / klinesC.close - 1) * 1000
    klinesC['peak3_bias'] = (klinesC.peak3 / klinesC.close - 1) * 1000
    klinesC['peak4_bias'] = (klinesC.peak4 / klinesC.close - 1) * 1000
 
    klinesC['valley1_bias'] = (klinesC.valley1 / klinesC.close - 1) * 1000
    klinesC['valley2_bias'] = (klinesC.valley2 / klinesC.close - 1) * 1000
    klinesC['valley3_bias'] = (klinesC.valley3 / klinesC.close - 1) * 1000
    klinesC['valley4_bias'] = (klinesC.valley4 / klinesC.close - 1) * 1000   
    
    klinesC['peak1_prev1_bias'] = (klinesC.p_prev1_1 / klinesC.close - 1) * 1000
    klinesC['peak2_prev1_bias'] = (klinesC.p_prev1_2 / klinesC.close - 1) * 1000
    klinesC['peak3_prev1_bias'] = (klinesC.p_prev1_3 / klinesC.close - 1) * 1000
    klinesC['peak4_prev1_bias'] = (klinesC.p_prev1_4 / klinesC.close - 1) * 1000

    klinesC['valley1_prev1_bias'] = (klinesC.v_prev1_1 / klinesC.close - 1) * 1000
    klinesC['valley2_prev1_bias'] = (klinesC.v_prev1_2 / klinesC.close - 1) * 1000
    klinesC['valley3_prev1_bias'] = (klinesC.v_prev1_3 / klinesC.close - 1) * 1000
    klinesC['valley4_prev1_bias'] = (klinesC.v_prev1_4 / klinesC.close - 1) * 1000  
    
    klinesC['tl_price'] = get_yc_vol_price(klinesC[['close']],klinesC.v3_strength,3)
    klinesC['jl_price'] = get_yc_vol_price(klinesC[['close']],klinesC.v3_strength,1.5)
    
#     klinesC['stop_loss_long'] = gen_stop_loss_long(klinesC,para)  # move to prepare_back_test_data
#     get_nearest_supp_resis(klinesC[['close']],klinesC[['valley1','valley2','valley3','valley_last1','valley_last2','valley_last3',\
#                     'peak1','peak2','peak3','peak_last1','peak_last2','peak_last3']])
    return klinesC

# 交易

In [ ]:
# long only
def get_long_order(source):
    data = source.copy()
    position = 0
    for k in range(len(data)):
        if data.BBB[k] and position == 0:
            data.orders.values[k] = 1
            position = 1
        elif position == 1  and ((data.relative_h_bsp[k]) or (data.cross_b_stoploss[k])):
            data.orders.values[k] = -1
            position = 0

In [ ]:
def get_heavy_bsp(kline):
    return abs(kline.BIAS1) > 6 and kline.relative_bsp > 4

In [ ]:
def get_cross_b_stoploss(kline):
    return kline.close < kline.B_STOPLOSS

In [ ]:
def get_cross_s_stoploss(kline):
    return kline.close > kline.S_STOPLOSS